# recursive_opt — Structured Use-Case Experiment Suite (controlled refactor)

This notebook is a **controlled-structure rewrite** of `examples/recursive_opt_use_cases.ipynb`.

Goals of this version:

1. keep **all current UC families** (UC1–UC14, plus three-way / Stage 2 / tier follow-ups),
2. make **definitions vs executions** explicit,
3. centralize reusable notebook-only helpers,
4. avoid silent coupling between cells,
5. show **clearly what is runnable by default**, what is optional, and what is expensive.

**Important controlled-change rule**

- No package file is modified here.
- Any helper that should eventually move out of the notebook stays in a notebook cell first, with an English comment telling where it should go later.
- Artifact/result formats stay compatible with the existing `examples/notebook_outputs/recursive_opt_use_cases` layout.


## Target tree (minimal-risk target structure)

```text
examples/
└── recursive_opt_use_cases_structured.ipynb   # this notebook only

Notebook internal structure
├── 00. execution plan / safety switches
├── 01. configuration / mode banner
├── 02. shared harness / result rendering
├── 03. notebook-local reusable helpers
│   ├── generic spec builders
│   ├── generic code/config runners
│   ├── generic guarded-policy utilities
│   └── historical summary/export helpers
├── 04. diagnostics
├── 05. UC definitions
│   ├── UC1–UC4
│   ├── UC5–UC7
│   ├── UC8–UC11
│   └── UC12–UC14
├── 06. explicit UC execution cells
├── 07. three-way benchmarks
├── 08. Stage 2 / guarded variants / budget sweeps / tier follow-ups
└── 09. summaries / exports / historical scans
```

This keeps the **package tree unchanged** and limits the refactor to a single notebook file.


## Execution model

This notebook separates:

- **definitions**: functions, evaluators, spec builders, baselines,
- **executions**: cells that actually materialize `uc1`, `uc2`, ..., `uc14`,
- **optional heavy sections**: three-way, Stage 2, tier follow-ups.

Default plan:

- run diagnostics,
- run UC1–UC13 single-arm notebook experiments,
- skip UC14, three-way, Stage 2, guarded variants, and tier follow-ups unless explicitly enabled,
- always allow summaries/historical scans.

That makes it obvious what has run in the current kernel.


In [ ]:
# ============================ 00. EXECUTION PLAN =============================
RUN = {
    "diagnostics": True,
    "uc1": True,
    "uc2": True,
    "uc3": True,
    "uc4": True,
    "uc5": True,
    "uc6": True,
    "uc7": True,
    "uc8": True,
    "uc9": True,
    "uc10": True,
    "uc11": True,
    "uc12": True,
    "uc13": True,
    "uc14": False,
    "guarded_variants": False,
    "three_way": False,
    "stage2": False,
    "tier_followups": False,
    "summaries": True,
    "historical": True,
    "exports": True,
}

def enabled(name: str) -> bool:
    return bool(RUN.get(name, False))

print("Execution plan:")
for _k in sorted(RUN):
    print(f"  - {_k:16s} = {RUN[_k]}")


In [ ]:

# ============================ 01. CONFIGURATION ==============================
import os, sys, json, time, statistics, textwrap, math
from pathlib import Path
from IPython.display import Markdown, display

_REPO_ROOT = Path.cwd()
if not (_REPO_ROOT / "opto").exists() and (_REPO_ROOT.parent / "opto").exists():
    _REPO_ROOT = _REPO_ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

REQUESTED_LIVE = os.environ.get("RECURSIVE_OPT_LIVE", "0").strip().lower() not in {"0", "false", "off", "no"}
HAVE_KEY = any(os.environ.get(k) for k in ("OPENAI_API_KEY", "OPENROUTER_API_KEY", "OPENAI_ADMIN_KEY"))
LIVE = bool(REQUESTED_LIVE and HAVE_KEY)
if REQUESTED_LIVE and not HAVE_KEY:
    raise RuntimeError(
        "RECURSIVE_OPT_LIVE=1 but no API key is set. "
        "Set OPENAI_API_KEY / OPENROUTER_API_KEY, or run in offline mode."
    )

MODEL = os.environ.get("RECURSIVE_OPT_MODEL") or os.environ.get("TRACE_LITELLM_MODEL") or "gpt-5.4-nano"
os.environ["RECURSIVE_OPT_MODEL"] = MODEL
os.environ["TRACE_LITELLM_MODEL"] = MODEL

RECURSIVE_OPTIMIZER = os.environ.get("RECURSIVE_OPT_OPTIMIZER", "OptoPrimeV2")
RECURSIVE_OPTIMIZER_KWARGS = json.loads(os.environ.get("RECURSIVE_OPT_OPTIMIZER_KWARGS", "{}"))
RECURSIVE_LLM_PROFILES = json.loads(os.environ.get("RECURSIVE_OPT_LLM_PROFILES", "{}"))

TRACEBENCH_OPTIMIZER = os.environ.get("RECURSIVE_OPT_TRACEBENCH_OPTIMIZER", "OptoPrimeV2")
TRACEBENCH_OPTIMIZER_KWARGS = json.loads(os.environ.get("RECURSIVE_OPT_TRACEBENCH_OPTIMIZER_KWARGS", "{}"))
if TRACEBENCH_OPTIMIZER == "OptoPrimeMultiV2" and not TRACEBENCH_OPTIMIZER_KWARGS:
    TRACEBENCH_OPTIMIZER_KWARGS = {
        "num_responses": 4,
        "generation_technique": "multi_experts",
        "selection_technique": "random",
        "experts_list": ["Algorithm Expert", "Performance Optimizer", "Prompt Engineer", "Critical Reviewer"],
    }
os.environ["RECURSIVE_OPT_TRACEBENCH_OPTIMIZER_KWARGS"] = json.dumps(TRACEBENCH_OPTIMIZER_KWARGS)
os.environ["RECURSIVE_OPT_OPTIMIZER"] = RECURSIVE_OPTIMIZER
os.environ["RECURSIVE_OPT_OPTIMIZER_KWARGS"] = json.dumps(RECURSIVE_OPTIMIZER_KWARGS)
os.environ["RECURSIVE_OPT_LLM_PROFILES"] = json.dumps(RECURSIVE_LLM_PROFILES)

WALL_TIME_S           = int(os.environ.get("RECURSIVE_OPT_WALL_TIME_S", "1800"))
RUN_ITERATIONS        = int(os.environ.get("RECURSIVE_OPT_ITERATIONS", "2"))
NUM_CANDIDATES        = int(os.environ.get("RECURSIVE_OPT_NUM_CANDIDATES", "2"))
MAX_OPTIMIZER_CALLS   = int(os.environ.get("RECURSIVE_OPT_MAX_OPTIMIZER_CALLS", "8"))
MAX_EVAL_CALLS        = int(os.environ.get("RECURSIVE_OPT_MAX_EVAL_CALLS", "48"))
CAPABILITY_EVAL_CALLS = int(os.environ.get("RECURSIVE_OPT_CAPABILITY_EVAL_CALLS", "96"))
MAX_CANDIDATES        = int(os.environ.get("RECURSIVE_OPT_MAX_CANDIDATES", "8"))

MAX_EXAMPLES       = int(os.environ.get("RECURSIVE_OPT_MAX_EXAMPLES", "8"))
HARD_MAX_EXAMPLES  = int(os.environ.get("RECURSIVE_OPT_HARD_MAX_EXAMPLES", "4"))
INNER_STEPS        = int(os.environ.get("RECURSIVE_OPT_INNER_STEPS", "0"))
TIMEOUT_S          = int(os.environ.get("RECURSIVE_OPT_TIMEOUT_S", "35"))

SEEDS = [0, 1, 2, 3, 4]
DIAGNOSTIC_SEEDS = list(SEEDS)

RUN_ID = os.environ.get("RECURSIVE_OPT_RUN_ID") or time.strftime("use_cases_structured_%Y%m%d_%H%M%S")
_DEFAULT_OUTPUT_ROOT = _REPO_ROOT / "examples/notebook_outputs/recursive_opt_use_cases"
OUTPUT_ROOT = Path(os.environ.get("RECURSIVE_OPT_OUTPUT_ROOT", str(_DEFAULT_OUTPUT_ROOT))) / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

os.environ["RECURSIVE_OPT_ITERATIONS"] = str(RUN_ITERATIONS)
os.environ["RECURSIVE_OPT_NUM_CANDIDATES"] = str(NUM_CANDIDATES)
os.environ["RECURSIVE_OPT_CAPABILITY_MAX_EXAMPLES"] = str(MAX_EXAMPLES)

if LIVE:
    from opto.features.recursive_opt.runmode import preflight_model
    from opto.features.recursive_opt.tracebench import ensure_default_task_adapter
    preflight_model(MODEL)
    ensure_default_task_adapter(require=True)

print(
    "LIVE =", LIVE,
    "| requested_live =", REQUESTED_LIVE,
    "| model =", MODEL,
    "| tracebench_optimizer =", TRACEBENCH_OPTIMIZER,
    "| recursive_optimizer =", RECURSIVE_OPTIMIZER,
    "| recursive_llm_profiles =", RECURSIVE_OPTIMIZER_KWARGS.get("llm_profiles", []),
    "| registered_llm_profiles =", sorted(RECURSIVE_LLM_PROFILES),
    "| iterations =", RUN_ITERATIONS,
    "| candidates =", NUM_CANDIDATES,
    "| examples =", MAX_EXAMPLES,
    "| hard_examples =", HARD_MAX_EXAMPLES,
    "| seeds =", SEEDS,
    "| output_root =", OUTPUT_ROOT,
)


In [ ]:

# ===================== 02. SHARED HARNESS / RENDERING =======================
from opto.features.recursive_opt import run_spec, make_level_spec, MemoryLite
from opto.features.recursive_opt.budget import RecursiveOptBudget, reset_budget

def budget_block():
    return {
        "wall_time_s": WALL_TIME_S,
        "optimizer_llm_calls": MAX_OPTIMIZER_CALLS,
        "eval_llm_calls": MAX_EVAL_CALLS,
        "candidates": MAX_CANDIDATES,
        "on_exceed": "return_best",
    }

def make_budget():
    return RecursiveOptBudget(
        max_wall_time_s=WALL_TIME_S,
        max_optimizer_llm_calls=MAX_OPTIMIZER_CALLS,
        max_eval_llm_calls=MAX_EVAL_CALLS,
        max_candidates=MAX_CANDIDATES,
        stop_policy="return_best",
    )

def reset_standard_budget():
    reset_budget(make_budget())

def memory_path(name: str) -> str:
    safe = str(name).strip().strip("./") or "mem"
    return str(OUTPUT_ROOT / safe)

def write_experiment_json(root, filename, payload):
    path = Path(root) / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + "\n")
    return str(path)

def tracebench_block(max_examples=None, inner_steps=None, timeout_seconds=None, eval_kwargs=None, optimizer_kwargs=None):
    block = {
        "max_examples": int(max_examples or MAX_EXAMPLES),
        "inner_steps": INNER_STEPS if inner_steps is None else int(inner_steps),
        "timeout_seconds": TIMEOUT_S if timeout_seconds is None else int(timeout_seconds),
    }
    if eval_kwargs:
        block["eval_kwargs"] = dict(eval_kwargs)
    opt_kwargs = TRACEBENCH_OPTIMIZER_KWARGS if optimizer_kwargs is None else optimizer_kwargs
    if opt_kwargs:
        block["optimizer_kwargs"] = dict(opt_kwargs)
    return block

def _one_line_error(exc):
    lines = [line.strip() for line in str(exc).splitlines() if line.strip()]
    detail = lines[0][:160] if lines else repr(exc)[:160]
    return f"{type(exc).__name__}: {detail}"

def _finite(values):
    out = []
    for value in values:
        try:
            f = float(value)
        except (TypeError, ValueError):
            continue
        if math.isfinite(f):
            out.append(f)
    return out

def _fmt(value):
    if value is None:
        return "-"
    try:
        f = float(value)
    except (TypeError, ValueError):
        return str(value)
    return f"{f:.3f}" if math.isfinite(f) else "-"

def _md_cell(value):
    text = "-" if value is None else str(value)
    return text.replace("\n", "<br>").replace("|", "\\|")

def _md_code(value):
    text = _md_cell(value).replace("`", "\\`")
    return f"`{text}`"

def _compact_markdown_tables(markdown: str) -> str:
    lines = markdown.splitlines()
    compacted = []
    for index, line in enumerate(lines):
        if line.strip() == "" and compacted and compacted[-1].lstrip().startswith("|"):
            next_index = index + 1
            while next_index < len(lines) and lines[next_index].strip() == "":
                next_index += 1
            if next_index < len(lines) and lines[next_index].lstrip().startswith("|"):
                continue
        compacted.append(line)
    return "\n".join(compacted)

def _display_markdown(markdown: str) -> None:
    display(Markdown(_compact_markdown_tables(markdown)))

def _artifact_file(root):
    return str(Path(root) / "artifacts.jsonl")

def _artifact_ref(root, artifact_id=None):
    path = _artifact_file(root)
    return f"{path}#{artifact_id}" if artifact_id else path

def _turn_from_artifact_id(artifact_id):
    parts = str(artifact_id or "").split(":")
    if len(parts) < 3:
        return None
    try:
        return int(parts[-2])
    except (TypeError, ValueError):
        return None

def _artifact_turn(record):
    if not record:
        return None
    try:
        return int(record.get("iteration"))
    except (AttributeError, TypeError, ValueError):
        return _turn_from_artifact_id(record.get("artifact_id") if isinstance(record, dict) else None)

def _fmt_turn(value):
    if value is None:
        return "-"
    try:
        return str(int(value))
    except (TypeError, ValueError):
        return str(value)

def _best_step_from_progress(progress, key="best_objective_at"):
    if not isinstance(progress, dict):
        return None
    point = progress.get(key)
    if not isinstance(point, dict):
        return None
    return point.get("level_step")

def _artifact_version(result_or_row):
    if not isinstance(result_or_row, dict):
        return None
    version = result_or_row.get("artifact_version")
    if version is not None:
        return version
    return _turn_from_artifact_id(result_or_row.get("artifact_id") or result_or_row.get("artifact_file"))

def _result_mean(result):
    scores = _finite(result.get("scores", []))
    return statistics.mean(scores) if scores else None

def _result_delta(result):
    mean = _result_mean(result)
    initial = result.get("initial") if isinstance(result, dict) else None
    if mean is None or initial is None:
        return None
    try:
        return float(mean) - float(initial)
    except (TypeError, ValueError):
        return None

def _result_eval_calls(result):
    if not isinstance(result, dict):
        return None
    calls = result.get("eval_calls")
    if calls is not None:
        return calls
    progress = result.get("progress")
    if isinstance(progress, list):
        return len(progress)
    if isinstance(progress, dict) and isinstance(progress.get("history"), list):
        return len(progress["history"])
    return None


In [ ]:

# ================= 03. NOTEBOOK-LOCAL REUSABLE HELPERS ======================
def initial_score_for_spec(spec, level_id=None, run_name=None):
    if not LIVE:
        return None, None
    import opto.features.recursive_opt.spec as spec_mod
    try:
        reset_standard_budget()
        lid = level_id or spec["levels"][-1]["id"]
        base_root = Path(run_name or spec.get("memory_root", "mem")).name
        probe_spec = {**spec, "memory_root": memory_path(f"_initial_probes/{base_root}")}
        spec_mod.validate_spec(probe_spec)
        if "tracebench" in probe_spec:
            from opto.features.recursive_opt import tracebench as TB
            TB.configure_tracebench_adapter(probe_spec.get("tracebench") or {}, require=True)
        families = probe_spec.get("families", {})
        memory = MemoryLite(root=probe_spec["memory_root"])
        for level_spec in probe_spec["levels"]:
            level = spec_mod.compile_level(level_spec, memory, families, probe_spec.get("scoring"))
            if level_spec["id"] == lid:
                score, _data = spec_mod._final_eval(level, level_spec, families)
                score = spec_mod._clamp(score, spec_mod._clip_bounds(probe_spec.get("scoring")))
                return float(score), None
        return None, f"level {lid!r} not found"
    except Exception as exc:
        return None, _one_line_error(exc)

def run_spec_seeds(spec, seeds=SEEDS, level_id=None, run_name=None):
    scores, walls, artifact, aid, errors = [], [], None, None, []
    artifact_ref, best_score, best_step, artifact_version, best_progress = None, None, None, None, None
    lid = level_id or spec["levels"][-1]["id"]
    base_root = Path(run_name or spec.get("memory_root", "mem")).name
    spec_files, best_spec_file = [], None

    if not LIVE:
        for seed in seeds:
            root = memory_path(f"{base_root}_{seed}")
            run_spec_payload = {**spec, "memory_root": root}
            spec_files.append(write_experiment_json(root, "spec.json", run_spec_payload))
        if spec_files:
            best_spec_file = spec_files[0]
        return {
            "scores": [], "initial": None, "wall_s": None,
            "artifact": "(offline preflight: set LIVE=True to optimize)",
            "artifact_id": None, "artifact_file": None, "best_step": None,
            "artifact_version": None, "progress": None,
            "spec_file": best_spec_file, "dry": True, "errors": [],
        }

    initial, initial_error = initial_score_for_spec(spec, level_id=lid, run_name=base_root)
    if initial_error:
        errors.append(f"initial: {initial_error}")

    for seed in seeds:
        try:
            reset_standard_budget()
            root = memory_path(f"{base_root}_{seed}")
            run_spec_payload = {**spec, "memory_root": root}
            spec_file = write_experiment_json(root, "spec.json", run_spec_payload)
            spec_files.append(spec_file)
            out = run_spec(run_spec_payload)
            r = out["results"][lid]
            score = float(r["score"])
            scores.append(score)
            walls.append(float(r["wall_s"]))
            ref = _artifact_ref(root, r.get("artifact_id"))
            if best_score is None or score > best_score:
                best_score = score
                artifact, aid, artifact_ref = r["artifact"], r.get("artifact_id"), ref
                best_progress = r.get("progress") or {}
                best_step = _best_step_from_progress(best_progress)
                artifact_version = _turn_from_artifact_id(r.get("artifact_id"))
                best_spec_file = spec_file
        except Exception as exc:
            errors.append(f"seed {seed}: {_one_line_error(exc)}")

    return {
        "scores": scores,
        "initial": initial,
        "wall_s": round(statistics.mean(walls), 1) if walls else None,
        "artifact": artifact or "(no successful seed)",
        "artifact_id": aid,
        "artifact_file": artifact_ref,
        "best_step": best_step,
        "artifact_version": artifact_version,
        "progress": best_progress,
        "spec_file": best_spec_file or (spec_files[-1] if spec_files else None),
        "errors": errors,
        "dry": False,
    }

def _notes_for_result(result):
    notes = []
    if result.get("control_reason"):
        notes.append(str(result["control_reason"]))
    if result.get("notes"):
        notes.append(str(result["notes"]))
    artifact = result.get("artifact")
    if artifact and "best_config=" in str(artifact):
        notes.append(str(artifact))
    errors = list(result.get("errors", []) or [])
    notes.extend(str(error) for error in errors[:2])
    if len(errors) > 2:
        notes.append(f"+{len(errors)-2} more")
    return "; ".join(notes)

def summarize(rows):
    head = (
        "| experiment | initial | mean score | delta | std | n | wall_s | eval/trials | best step | artifact version | best artifact file | spec file | notes |\n"
        "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|---|---|"
    )
    lines = [head]
    for label, r in rows:
        if r.get("dry"):
            lines.append(f"| {_md_cell(label)} | - | offline | - | - | 0 | - | - | - | - | - | - | set LIVE=True |")
            continue
        scores = _finite(r.get("scores", []))
        mean = statistics.mean(scores) if scores else None
        std = statistics.pstdev(scores) if len(scores) > 1 else None
        delta = (mean - r["initial"]) if mean is not None and r.get("initial") is not None else None
        notes = _notes_for_result(r)
        artifact_file = r.get("artifact_file") or "-"
        spec_file = r.get("spec_file") or "-"
        lines.append(
            f"| {_md_cell(label)} | {_fmt(r.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
            f"{_fmt(std)} | {len(scores)} | {_fmt(r.get('wall_s'))} | {_fmt_turn(_result_eval_calls(r))} | "
            f"{_fmt_turn(r.get('best_step'))} | {_fmt_turn(_artifact_version(r))} | {_md_code(artifact_file)} | "
            f"{_md_code(spec_file)} | {_md_cell(notes)} |"
        )
    return "\n".join(lines)

def mark_control(result, reason):
    out = dict(result)
    out["exclude_best"] = True
    out["control_reason"] = reason
    return out

def best_of(rows):
    scored = [(l, r) for l, r in rows if _result_mean(r) is not None]
    informative = [(l, r) for l, r in scored if not r.get("exclude_best")]
    if informative:
        scored = informative
    if not scored:
        return None
    def key(row):
        _label, result = row
        mean = _result_mean(result)
        initial = result.get("initial")
        delta = mean - initial if mean is not None and initial is not None else 0.0
        return (mean, delta, -(result.get("wall_s") or 1e9))
    return max(scored, key=key)

def best_gain_of(rows):
    candidates = []
    for label, result in rows:
        if result.get("exclude_best"):
            continue
        delta = _result_delta(result)
        if delta is not None and delta > 0:
            candidates.append((label, result))
    if not candidates:
        return None
    def key(row):
        _label, result = row
        return (_result_delta(result) or 0.0, _result_mean(result) or float("-inf"), -(result.get("wall_s") or 1e9))
    return max(candidates, key=key)

def show_table(title, rows):
    display(Markdown(f"### {title}\n" + summarize(rows)))
    b = best_of(rows)
    g = best_gain_of(rows)
    if b:
        _display_markdown(
            f"**Best final score: `{_md_cell(b[0])}`** "
            f"— best step: `{_fmt_turn(b[1].get('best_step'))}` "
            f"— artifact version: `{_fmt_turn(_artifact_version(b[1]))}` "
            f"— artifact file: {_md_code(b[1].get('artifact_file') or '-')} "
            f"— spec file: {_md_code(b[1].get('spec_file') or '-')}"
        )
        print(textwrap.shorten(str(b[1]["artifact"]), 1200, placeholder=" ...[truncated]"))
    if g and (not b or g[0] != b[0]):
        _display_markdown(
            f"**Best positive non-control gain: `{_md_cell(g[0])}`** "
            f"— delta: `{_fmt(_result_delta(g[1]))}` "
            f"— artifact file: {_md_code(g[1].get('artifact_file') or '-')}"
        )

uc1 = []; uc2 = []; uc3 = []; uc4 = []; uc5 = []; uc6 = []; uc7 = []
uc8 = []; uc9 = []; uc10 = []; uc11 = []; uc12 = []; uc13 = []; uc14 = []
uc8_guarded = []; uc11_guarded = []
tw_uc1 = tw_uc2 = tw_uc4 = tw_uc5 = tw_uc8 = tw_uc9 = tw_uc10 = tw_uc11 = tw_uc13 = tw_uc14 = None
print("Shared notebook-local helpers ready.")


In [ ]:

# ================= 03b. HISTORICAL SCAN / EXPORT HELPERS ====================
def _read_jsonl(path):
    p = Path(path)
    if not p.exists():
        return []
    text = p.read_text()
    if p.suffix == ".json":
        try:
            data = json.loads(text)
        except json.JSONDecodeError:
            return []
        if isinstance(data, list):
            return [item for item in data if isinstance(item, dict)]
        return [data] if isinstance(data, dict) else []
    rows = []
    for line in text.splitlines():
        if line.strip():
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(item, dict):
                rows.append(item)
    return rows

def _best_artifact_from_dir(mem_dir):
    records = _read_jsonl(Path(mem_dir) / "artifacts.jsonl")
    valid = []
    for record in records:
        score = _finite([record.get("score")])
        if score:
            valid.append(record)
    return max(valid, key=lambda r: float(r["score"])) if valid else None

def _best_step_from_artifact(record):
    metrics = record.get("metrics") if isinstance(record, dict) else None
    if not isinstance(metrics, dict):
        return None
    return _best_step_from_progress(metrics.get("progress"))

def _initial_from_dir(mem_dir):
    for filename in ("artifacts.jsonl", "episodes.jsonl"):
        records = _read_jsonl(Path(mem_dir) / filename)
        for record in records:
            metrics = record.get("metrics") if isinstance(record, dict) else None
            if isinstance(metrics, dict):
                history = metrics.get("score_history")
                if isinstance(history, list):
                    scores = _finite(history[:1])
                    if scores:
                        return scores[0]
                scores = _finite([metrics.get("initial")])
                if scores:
                    return scores[0]
            scores = _finite([record.get("score")])
            if scores:
                return scores[0]
    return None

UC_DIR_PREFIXES = {
    "UC1 component code": "mem_uc1",
    "UC2 setup/config": "mem_uc2",
    "UC3 capability": "mem_uc3",
    "UC4 family/transfer": "mem_uc4",
    "UC5 optimizer/tool": "mem_uc5",
    "UC6 trace feedback": "mem_uc6",
    "UC7 graph/suboptimizer": ("mem_suboptimizer_graph", "mem_conditional_suboptimizer_graph"),
    "UC8 campaign policy": "mem_uc8",
    "UC9 agentic trace policy": "mem_uc9",
    "UC10 promotion policy": "mem_uc10",
    "UC11 prompt emitter": "mem_uc11",
    "UC12 promoted primitives": "mem_uc12",
    "UC13 numeric config": "mem_uc13",
}

def _uc_prefixes(prefix):
    return tuple(prefix) if isinstance(prefix, (list, tuple)) else (prefix,)

def summarize_past_runs(base_dir=None):
    base = Path(base_dir or OUTPUT_ROOT.parent)
    rows = []
    if not base.exists():
        return rows
    for run_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        for uc_name, prefix in UC_DIR_PREFIXES.items():
            mem_dirs = sorted(
                p
                for item in _uc_prefixes(prefix)
                for p in run_dir.glob(f"{item}*")
                if p.is_dir()
            )
            if not mem_dirs:
                continue
            best, best_dir = None, None
            finals, initials = [], []
            for mem_dir in mem_dirs:
                init = _initial_from_dir(mem_dir)
                if init is not None:
                    initials.append(init)
                art = _best_artifact_from_dir(mem_dir)
                if art is None:
                    continue
                score = float(art["score"])
                finals.append(score)
                if best is None or score > float(best["score"]):
                    best, best_dir = art, mem_dir
            if best is None:
                continue
            rows.append({
                "run": run_dir.name,
                "use_case": uc_name,
                "initial_mean": statistics.mean(initials) if initials else None,
                "best_score": float(best["score"]),
                "final_mean": statistics.mean(finals) if finals else None,
                "n_dirs": len(mem_dirs),
                "best_step": _best_step_from_artifact(best),
                "artifact_version": _artifact_turn(best),
                "artifact_file": _artifact_ref(best_dir, best.get("artifact_id")),
            })
    return rows

def _experiment_from_mem_dir(mem_dir):
    name = Path(mem_dir).name
    parts = name.split("_")
    if parts and parts[-1].isdigit():
        name = "_".join(parts[:-1])
    return name

def summarize_past_experiments(base_dir=None):
    base = Path(base_dir or OUTPUT_ROOT.parent)
    rows = []
    if not base.exists():
        return rows
    for run_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        for uc_name, prefix in UC_DIR_PREFIXES.items():
            for mem_dir in sorted(
                p
                for item in _uc_prefixes(prefix)
                for p in run_dir.glob(f"{item}*")
                if p.is_dir()
            ):
                art = _best_artifact_from_dir(mem_dir)
                if art is None:
                    continue
                rows.append({
                    "run": run_dir.name,
                    "use_case": uc_name,
                    "experiment": _experiment_from_mem_dir(mem_dir),
                    "initial": _initial_from_dir(mem_dir),
                    "best_score": float(art["score"]),
                    "best_step": _best_step_from_artifact(art),
                    "artifact_version": _artifact_turn(art),
                    "artifact_file": _artifact_ref(mem_dir, art.get("artifact_id")),
                })
    return rows

def past_experiments_table(rows, limit=None):
    head = "| run | use case | experiment | initial | best score | best step | artifact version | best artifact file |\n|---|---|---|---:|---:|---:|---:|---|"
    lines = [head]
    ordered = sorted(rows, key=lambda r: (r["run"], r["use_case"], r["experiment"]))
    selected = ordered if limit is None else ordered[-limit:]
    for row in selected:
        lines.append(
            f"| {_md_cell(row['run'])} | {_md_cell(row['use_case'])} | {_md_cell(row['experiment'])} | "
            f"{_fmt(row['initial'])} | {_fmt(row['best_score'])} | {_fmt_turn(row.get('best_step'))} | "
            f"{_fmt_turn(_artifact_version(row))} | {_md_code(row['artifact_file'])} |"
        )
    if limit is not None and len(ordered) > limit:
        lines.append(f"| ... | ... | {len(ordered)-limit} older rows omitted | - | - | - | - | - |")
    return "\n".join(lines)

def past_runs_table(rows):
    head = "| run | use case | initial mean | final mean | best score | best step | artifact version | n memory dirs | best artifact file |\n|---|---|---:|---:|---:|---:|---:|---:|---|"
    lines = [head]
    for row in rows:
        lines.append(
            f"| {_md_cell(row['run'])} | {_md_cell(row['use_case'])} | {_fmt(row['initial_mean'])} | "
            f"{_fmt(row['final_mean'])} | {_fmt(row['best_score'])} | {_fmt_turn(row.get('best_step'))} | "
            f"{_fmt_turn(_artifact_version(row))} | {row['n_dirs']} | {_md_code(row['artifact_file'])} |"
        )
    return "\n".join(lines)

def _sanitize_export_name(value):
    import re
    text = re.sub(r"[^a-zA-Z0-9_.-]+", "_", str(value).strip().lower())
    return text.strip("._")[:80] or "artifact"

def _record_from_artifact_ref(ref):
    if not ref or ref == "-":
        return None
    file_part, sep, artifact_id = str(ref).partition("#")
    path = Path(file_part)
    records = _read_jsonl(path)
    if not records:
        return None
    if sep:
        for record in records:
            if record.get("artifact_id") == artifact_id:
                return record
    scored = [(score[0], record) for record in records if (score := _finite([record.get("score")]))]
    return max(scored, key=lambda item: item[0])[1] if scored else records[-1]

def _artifact_suffix(kind, content):
    if kind == "code":
        return ".py"
    if kind == "graph":
        return ".json"
    return ".txt"

def export_best_artifacts(all_results):
    out_dir = OUTPUT_ROOT / "best_artifacts"
    out_dir.mkdir(parents=True, exist_ok=True)
    index = []
    for number, (use_case, rows) in enumerate(all_results.items(), start=1):
        best = best_of(rows)
        if best is None:
            continue
        label, result = best
        record = _record_from_artifact_ref(result.get("artifact_file"))
        content = record.get("content") if isinstance(record, dict) else result.get("artifact")
        if content is None:
            continue
        kind = record.get("kind") if isinstance(record, dict) else "artifact"
        suffix = _artifact_suffix(kind, content)
        stem = f"{number:02d}_{_sanitize_export_name(use_case)}__{_sanitize_export_name(label)}"
        path = out_dir / f"{stem}{suffix}"
        if isinstance(content, (dict, list)):
            path.write_text(json.dumps(content, indent=2, sort_keys=True) + "\n")
        else:
            path.write_text(str(content).rstrip() + "\n")
        mean = _result_mean(result)
        index.append({
            "use_case": use_case,
            "experiment": label,
            "kind": kind,
            "score": record.get("score") if isinstance(record, dict) else mean,
            "initial": result.get("initial"),
            "mean_score": mean,
            "artifact_ref": result.get("artifact_file"),
            "export_file": str(path),
            "spec_file": result.get("spec_file"),
        })
    (out_dir / "index.json").write_text(json.dumps(index, indent=2, sort_keys=True, default=str) + "\n")
    return index

def _artifact_exports_table(index):
    head = "| use case | experiment | kind | score | export file | source artifact |\n|---|---|---|---:|---|---|"
    lines = [head]
    for item in index:
        lines.append(
            f"| {_md_cell(item['use_case'])} | {_md_cell(item['experiment'])} | {_md_cell(item['kind'])} | "
            f"{_fmt(item['score'])} | {_md_code(item['export_file'])} | {_md_code(item['artifact_ref'])} |"
        )
    return "\n".join(lines)


## 04. Root-cause diagnostics

This section is intentionally small and cheap.

It answers three questions before spending serious budget:

1. does the task/surface have **score spread**?
2. is a code-surface baseline already **saturated**?
3. are we mixing a prompt surface with a task that is actually a **raw/code artifact** task?


In [ ]:

from opto.features.recursive_opt.spec import score_spread
from opto.features.recursive_opt.tracebench import make_code_evaluator, configure_tracebench_adapter

if enabled("diagnostics"):
    if LIVE:
        configure_tracebench_adapter(tracebench_block(), require=True)
        diagnostic_rows = []
        probe_prompts = [
            {},
            {"starting_artifact": "Answer directly."},
            {"starting_artifact": "Plan step by step, then verify the answer before replying."},
        ]
        for task in ["internal:multiobjective_gsm8k", "internal:multiobjective_bbeh", "hf:drop", "hf:qasper"]:
            try:
                spread = score_spread(task, probes=probe_prompts)
                scores = [r.get("score") for r in spread["rows"]]
                diagnostic_rows.append((task, spread["valid_spread"], spread["invalid_probes"], scores))
            except Exception as exc:
                diagnostic_rows.append((task, None, None, _one_line_error(exc)))

        code_probe = make_code_evaluator("internal:batch_design", "batch_design")
        code_rows = []
        for label, fn in [
            ("take_first", lambda n, k: list(range(k))),
            ("take_last", lambda n, k: list(range(n-k, n))),
            ("stride", lambda n, k: list(range(0, n, max(1, n//k)))[:k]),
            ("hard_mod3", lambda n, k: [i for i in range(n) if i % 3 == 0][:k]),
        ]:
            score, feedback = code_probe(lambda **kw: fn(**kw), "internal:batch_design")
            code_rows.append((label, score, feedback))

        lines = ["| probe | spread/score | details |", "|---|---:|---|"]
        for task, spread, invalid, scores in diagnostic_rows:
            lines.append(f"| {task} score spread | {_fmt(spread)} | invalid={invalid}; scores={scores} |")
        for label, score, feedback in code_rows:
            lines.append(f"| batch_design baseline `{label}` | {_fmt(score)} | {feedback[:180]} |")
        display(Markdown("\n".join(lines)))
    else:
        display(Markdown("Diagnostics skipped: set `RECURSIVE_OPT_LIVE=1` to run live diagnostics."))
else:
    print("Diagnostics disabled by RUN['diagnostics']=False")


In [ ]:

# ================= 05. GENERIC BUILDERS / SPEC HELPERS ======================
from opto.features.recursive_opt import CodeArtifactLevel, ComponentSpec, optimize, RecursiveGuide
from opto.features.recursive_opt.tracebench import (
    make_code_evaluator,
    make_dataset,
    make_tracebench_direct_answer_evaluator,
    make_artifact_emitter_evaluator,
    make_multiobjective_evaluator,
)

def run_code_experiment(
    name,
    task_id,
    objective,
    *,
    seeds=SEEDS,
    memory_name=None,
    baseline=None,
    evaluate=None,
    iterations=None,
    num_candidates=None,
):
    scores, initial_scores, walls, final_code, errors = [], [], [], None, []
    best_ref, best_score, best_code, best_spec_file, artifact_version = None, None, None, None, None
    for seed in seeds:
        root_name = memory_name or f"mem_uc1_{name}"
        root = memory_path(f"{root_name}_{seed}")
        baseline_fn = baseline or _BASELINES[name]
        local_iterations = int(iterations or RUN_ITERATIONS)
        local_candidates = int(num_candidates or NUM_CANDIDATES)
        payload = {
            "surface": "code",
            "component": name,
            "task_id": task_id,
            "objective": objective,
            "baseline": getattr(baseline_fn, "__name__", str(baseline_fn)),
            "iterations": local_iterations,
            "num_candidates": local_candidates,
            "max_examples": MAX_EXAMPLES,
        }
        spec_file = write_experiment_json(root, "component_spec.json", payload)
        best_spec_file = spec_file
        if not LIVE:
            continue
        try:
            mem = MemoryLite(root=root)
            spec = ComponentSpec(name=name, baseline=baseline_fn, evaluate=evaluate or make_code_evaluator(task_id, name), objective=objective)
            level = CodeArtifactLevel(spec, memory=mem)
            guide = RecursiveGuide()
            initial_scores.append(float(guide(task_id, level.forward(task_id), None)[0]))
            reset_standard_budget()
            t0 = time.time()
            optimize(level, make_dataset([task_id], repeats=MAX_EXAMPLES), guide=guide, iterations=local_iterations, num_candidates=local_candidates)
            best = mem.best_artifact(str(task_id), "code")
            if best is not None and level.parameters():
                level.parameters()[0]._data = best.content
            walls.append(round(time.time() - t0, 1))
            score = float(guide(task_id, level.forward(task_id), None)[0])
            if best is not None and float(best.score) >= score:
                score, final_code = float(best.score), best.content
                ref = _artifact_ref(root, best.artifact_id)
                turn = int(best.iteration)
            else:
                final_code = level.current_code()
                ref = _artifact_file(root)
                turn = _turn_from_artifact_id(ref)
            scores.append(score)
            if best_score is None or score > best_score:
                best_score, best_ref, best_code, best_spec_file = score, ref, final_code, spec_file
                artifact_version = turn
        except Exception as exc:
            errors.append(f"seed {seed}: {_one_line_error(exc)}")
    if not LIVE:
        return {"scores": [], "initial": None, "wall_s": None, "artifact": "(offline preflight) set LIVE=True to optimize", "artifact_id": None, "artifact_file": None, "best_step": None, "artifact_version": None, "progress": None, "spec_file": best_spec_file, "dry": True, "errors": errors}
    return {"scores": scores, "initial": statistics.mean(initial_scores) if initial_scores else None, "wall_s": round(statistics.mean(walls), 1) if walls else None, "artifact": best_code or final_code or "(no successful seed)", "artifact_id": None, "artifact_file": best_ref, "best_step": None, "artifact_version": artifact_version, "progress": None, "spec_file": best_spec_file, "errors": errors, "dry": False}

def numeric_search_space(fields, constraints):
    return {field: ("cat", tuple(constraints[field])) for field in fields if field in constraints}

def numeric_optimizer_arm(task, fields, constraints, *, tasks=None, inner_steps=2, max_examples=6, family_name="numeric_arm", trials=16, memory_root="./mem_numeric_arm"):
    base = {"scores": [], "initial": None, "wall_s": None, "artifact": None, "artifact_id": None, "artifact_file": None, "best_step": None, "artifact_version": None, "progress": None, "spec_file": None, "eval_calls": None, "errors": [], "dry": False}
    if not LIVE:
        return {**base, "dry": True, "artifact": "(offline preflight: set LIVE=True to run the numeric optimizer)"}
    try:
        from opto.features.recursive_opt import optimize_config_numeric, MemoryLite
        from opto.features.recursive_opt import spec as _spec
        task_ids = list(tasks or [task])
        spec = config_spec(fields, numeric_constraints=constraints, task=task_ids[0], tasks=task_ids if len(task_ids) > 1 else None, family_name=family_name, max_examples=max_examples, inner_steps=inner_steps, memory_root=memory_root)
        fams = {family_name: task_ids}
        mem = MemoryLite(root=memory_path(memory_root + "_lvl"))
        level = _spec.compile_level(spec["levels"][0], mem, fams)
        eval_label = task_ids[0] if len(task_ids) == 1 else f"task_set:{family_name}"
        t0 = time.time()
        best, score, history = optimize_config_numeric(level, eval_label, fields, optimizer="optuna", max_trials=trials, space=numeric_search_space(fields, constraints))
        wall_s = round(time.time() - t0, 1)
        eval_calls = len(history)
        curve = [round(float(s), 3) for _, s in history]
        artifact_text = f"best_config={best} | curve={curve}"
        root = memory_path(memory_root + "_lvl")
        artifact_payload = {"best_config": best, "score": float(score), "curve": curve, "history": [{"assignment": assignment, "score": float(s)} for assignment, s in history], "fields": list(fields), "tasks": task_ids, "optimizer": "optuna", "trials": int(trials), "eval_calls": eval_calls, "wall_s": wall_s}
        artifact_file = write_experiment_json(root, "numeric_optimizer_result.json", artifact_payload)
        spec_file = write_experiment_json(root, "spec.json", spec)
        return {**base, "scores": [float(score)], "initial": curve[0] if curve else None, "artifact": artifact_text, "artifact_file": artifact_file, "wall_s": wall_s, "eval_calls": eval_calls, "best_step": (max(range(len(curve)), key=lambda k: curve[k]) if curve else None), "progress": curve, "spec_file": spec_file, "notes": f"numeric trials={eval_calls}; zero LLM proposal calls; each trial still runs the real inner evaluator"}
    except Exception as exc:
        return {**base, "errors": [_one_line_error(exc)]}

FAMILY_TASK = "internal:multiobjective_gsm8k"
HARD_PROMPT_TASKS = {"drop": "hf:drop", "qasper": "hf:qasper"}
ART_MENU = ["", "Answer directly.", "Plan step by step, then answer.", "Plan step by step, then verify the answer before replying.", "Use the provided context as evidence, reason briefly, then answer exactly."]
CAUSAL_NUMERIC_TARGETS = ["batch_design", "batch_size"]
CAUSAL_NUMERIC_CONSTRAINTS = {"batch_design": ["random", "failure_balanced", "curriculum", "diversity"], "batch_size": [2, 4, 8]}

def config_spec(targets, reuse=False, extra_constraints=None, numeric_constraints=None, memory_root="./mem_uc2", task=FAMILY_TASK, tasks=None, family_name="reasoning", max_examples=None, inner_steps=None, fixed_overrides=None, budget=None):
    task_ids = list(tasks or [task])
    cons = {"starting_artifact": ART_MENU}
    if extra_constraints:
        cons.update(extra_constraints)
    if numeric_constraints:
        cons.update(numeric_constraints)
    fixed = {"optimizer": TRACEBENCH_OPTIMIZER, "trace_type": "internal", "credit_horizon": "step", "trainer": "PrioritySearch"}
    if fixed_overrides:
        fixed.update(fixed_overrides)
    level_kwargs = {"task": task_ids[0]} if len(task_ids) == 1 else {"tasks": task_ids}
    return {"families": {family_name: task_ids}, "memory_root": memory_root, "reuse_priors": reuse, "budget": dict(budget or budget_block()), "tracebench": tracebench_block(max_examples=max_examples, inner_steps=inner_steps), "scoring": {"clip": [-1.0, 1.0]}, "levels": [make_level_spec(id="o1_setup", surface="config", family=family_name, **level_kwargs, targets=targets, constraints=cons, fixed=fixed, iterations=RUN_ITERATIONS)]}

CAP_TASKS = ["internal:multiobjective_gsm8k"]
CAP_OBJECTIVES = {"accuracy": "max", "cost": "min"}
_cap_evaluator = make_multiobjective_evaluator(CAP_TASKS, CAP_OBJECTIVES, required_terms=("plan", "verify"))

def capability_spec(seed_text, memory_root="./mem_uc3"):
    cap_budget = {**budget_block(), "eval_llm_calls": CAPABILITY_EVAL_CALLS}
    return {"families": {"reasoning": CAP_TASKS}, "memory_root": memory_root, "budget": cap_budget, "tracebench": tracebench_block(), "levels": [make_level_spec(id="cap", surface="capability", family="reasoning", task=CAP_TASKS[0], seed=seed_text, evaluator=_cap_evaluator, objective_config={"mode": "pareto", "minimize": ["cost"]}, iterations=RUN_ITERATIONS)]}

def family_policy_spec(kind="o2", warm=False, targets=None, constraints=None, inner_steps=None, memory_root="./mem_uc4"):
    fams = {"gsm8k": [FAMILY_TASK], "qasper": [HARD_PROMPT_TASKS["qasper"]]}
    target_fields = list(targets or ["starting_artifact"])
    cons = dict(constraints or {"starting_artifact": ART_MENU})
    levels = [make_level_spec(id="o2_policy", surface="family_policy", family="*", families=list(fams), targets=target_fields, constraints=cons, fixed={"optimizer": TRACEBENCH_OPTIMIZER, "trainer": "PrioritySearch", "trace_type": "internal", "credit_horizon": "step"}, iterations=RUN_ITERATIONS)]
    if kind == "o3":
        levels.append(make_level_spec(id="o3_prior", surface="prior", family="*", task=HARD_PROMPT_TASKS["qasper"], targets=target_fields, constraints=cons, fixed={"optimizer": TRACEBENCH_OPTIMIZER, "trainer": "PrioritySearch", "trace_type": "internal", "credit_horizon": "step"}, iterations=RUN_ITERATIONS))
    return {"families": fams, "memory_root": memory_root, "reuse_priors": warm, "budget": budget_block(), "tracebench": tracebench_block(max_examples=HARD_MAX_EXAMPLES, inner_steps=inner_steps), "scoring": {"clip": [-1.0, 1.0]}, "levels": levels}

UC6_TASK = HARD_PROMPT_TASKS["qasper"]
def feedback_spec(level_id, trace_type, targets=None, constraints=None, inner_steps=None, memory_root=None):
    target_fields = list(targets or ["starting_artifact"])
    cons = dict(constraints or {"starting_artifact": ART_MENU})
    root = memory_root or f"./mem_uc6_{level_id}"
    return {"families": {"reasoning": [UC6_TASK]}, "memory_root": root, "budget": budget_block(), "tracebench": tracebench_block(max_examples=HARD_MAX_EXAMPLES, inner_steps=inner_steps), "scoring": {"clip": [-1.0, 1.0]}, "levels": [make_level_spec(id=level_id, surface="config", family="reasoning", task=UC6_TASK, targets=target_fields, constraints=cons, fixed={"optimizer": TRACEBENCH_OPTIMIZER, "trainer": "PrioritySearch", "trace_type": trace_type, "credit_horizon": "step"}, iterations=RUN_ITERATIONS)]}

def _weak_batch(self, n, k): return list(range(k))
def _trunc_summary(self, trace_text): return str(trace_text)[-500:]
def _bbeh_direct_solver(self, question): return "True"
def _norm_bool_answer(value): return str(value).strip().lower().replace(".", "").replace(" ", "")

_BASELINES = {"batch_design": _weak_batch, "trace_summarizer": _trunc_summary, "bbeh_direct_solver": _bbeh_direct_solver}


In [ ]:

# ===================== 05b. POLICY / DECISION UTILITIES =====================
from opto.features.recursive_opt import parse_optimizer_tool_policy, ConfidenceGate, GuardedDecisionCase, GuardedDecisionEvaluator

OPTIMIZER_TOOL_NAMES = ("trace_search", "run_subset", "artifact_linter", "note")

def _policy_text(raw):
    if isinstance(raw, dict):
        return json.dumps(raw, sort_keys=True).lower()
    return str(raw).lower()

def _mentioned_tasks(text, known_tasks):
    return {task for task in known_tasks if task.lower() in text}

def _keyword_present(text, word):
    import re
    key = str(word).strip().lower()
    if not key:
        return False
    if len(key) <= 5 or any(ch in key for ch in " _:/-"):
        return key in text
    return re.search(rf"(?<![a-z0-9_]){re.escape(key)}(?![a-z0-9_])", text) is not None

def _contains_any(text, words):
    return any(_keyword_present(text, word) for word in words)

def _max_examples_from_text(text):
    import re
    match = re.search(r"max_examples\s*[:=]\s*(\d+)", text)
    return int(match.group(1)) if match else None

def _baseline_take_last(self, n, k):  return list(range(n-k, n))
def _baseline_stride(self, n, k):     return list(range(0, n, max(1, n//k)))[:k]
def _baseline_tool_policy(self, signal): return "tools: note"

TOOL_POLICY_CASES = [
    {"signal": "Need prior failures and family examples before proposing a prompt update.", "required": {"trace_search", "note"}},
    {"signal": "Need validate a candidate on a small subset before accepting it.", "required": {"run_subset"}},
    {"signal": "Need inspect the saved artifact for syntax and current_code reuse.", "required": {"artifact_linter"}},
    {"signal": "Saturated control: record a note and do not spend expensive tool calls.", "required": {"note"}, "forbidden": {"trace_search", "run_subset", "artifact_linter"}},
]

def evaluate_optimizer_tool_policy(component, _task_id):
    scores, feedbacks = [], []
    for case in TOOL_POLICY_CASES:
        raw = component(case["signal"])
        selected = parse_optimizer_tool_policy(raw, OPTIMIZER_TOOL_NAMES, max_tools=3)
        selected_set = set(selected)
        required = set(case["required"])
        forbidden = set(case.get("forbidden", set()))
        extra = sorted(selected_set.difference(required).difference({"note"}))
        forbidden_hit = sorted(selected_set & forbidden)
        coverage = len(required & selected_set) / max(1, len(required))
        score = max(0.0, coverage - 0.15 * len(extra) - 0.35 * len(forbidden_hit))
        scores.append(score)
        feedbacks.append(f"selected={selected}; required={sorted(required)}; extra={extra}; forbidden_hit={forbidden_hit}; score={score:.2f}")
    return statistics.mean(scores), " | ".join(feedbacks)

_BASELINES["optimizer_tool_policy"] = _baseline_tool_policy

def _baseline_campaign_policy(self, diagnostics):
    return "action: continue\ntask: internal:multiobjective_gsm8k\nmax_examples: 8\nreason: default"

CAMPAIGN_TASKS = ("internal:multiobjective_gsm8k", "internal:multiobjective_bbeh", "hf:drop", "hf:qasper", "mixed:gsm8k+qasper")
CAMPAIGN_POLICY_CASES = [
    {"name": "saturated_drop_control", "diagnostics": {"task": "hf:drop", "mean_score": 1.0, "spread": 0.0, "recent_delta": 0.0, "wall_s": 38.0, "saturated": True}, "actions": {"stop", "skip", "control", "drop"}, "tasks": set(), "avoid": {"hf:drop"}, "max_examples": (0, 2), "reasons": {"satur", "ceiling", "control", "stop"}},
    {"name": "high_headroom_bbeh_exploit", "diagnostics": {"task": "internal:multiobjective_bbeh", "mean_score": 0.625, "spread": 1.0, "recent_delta": 0.375, "wall_s": 4.8, "saturated": False}, "actions": {"exploit", "train", "continue", "increase"}, "tasks": {"internal:multiobjective_bbeh"}, "avoid": set(), "max_examples": (8, 16), "reasons": {"headroom", "spread", "bbeh", "fast"}},
    {"name": "qasper_harder_probe", "diagnostics": {"task": "hf:qasper", "mean_score": 0.125, "spread": 0.082, "recent_delta": 0.037, "wall_s": 39.2, "saturated": False}, "actions": {"probe", "explore", "sample", "budget"}, "tasks": {"hf:qasper"}, "avoid": set(), "max_examples": (3, 6), "reasons": {"hard", "qasper", "noisy", "probe"}},
    {"name": "gsm8k_low_spread_stall", "diagnostics": {"task": "internal:multiobjective_gsm8k", "mean_score": -0.148, "spread": 0.042, "recent_delta": 0.002, "wall_s": 71.0, "saturated": False}, "actions": {"restart", "switch", "probe", "reduce"}, "tasks": {"hf:qasper", "internal:multiobjective_bbeh"}, "avoid": {"internal:multiobjective_gsm8k"}, "max_examples": (3, 8), "reasons": {"low", "spread", "stall", "switch"}},
    {"name": "mixed_regressed_split", "diagnostics": {"task": "mixed:gsm8k+qasper", "mean_score": -0.010, "spread": 0.008, "recent_delta": -0.006, "wall_s": 69.8, "mixed_regressed": True}, "actions": {"split", "separate", "restart", "ablate"}, "tasks": {"hf:qasper", "internal:multiobjective_bbeh"}, "avoid": {"mixed:gsm8k+qasper"}, "max_examples": (3, 8), "reasons": {"mixed", "regress", "separate", "ablate"}},
]

def evaluate_campaign_policy(component, _task_id):
    scores, feedbacks = [], []
    for case in CAMPAIGN_POLICY_CASES:
        raw = component(case["diagnostics"])
        text = _policy_text(raw)
        selected_tasks = _mentioned_tasks(text, CAMPAIGN_TASKS)
        max_examples = _max_examples_from_text(text)
        action_score = 1.0 if _contains_any(text, case["actions"]) else 0.0
        task_score = 1.0 if not case["tasks"] else min(1.0, len(selected_tasks & case["tasks"]) / len(case["tasks"]))
        avoid_score = 1.0 if not (selected_tasks & case["avoid"]) else 0.0
        budget_score = 0.0 if max_examples is None else (1.0 if case["max_examples"][0] <= max_examples <= case["max_examples"][1] else 0.0)
        reason_score = min(1.0, sum(1 for word in case["reasons"] if _keyword_present(text, word)) / 2.0)
        score = 0.30 * action_score + 0.25 * task_score + 0.20 * avoid_score + 0.15 * budget_score + 0.10 * reason_score
        scores.append(score)
        feedbacks.append(f"{case['name']}: score={score:.2f}; selected={sorted(selected_tasks)}; max_examples={max_examples}")
    return statistics.mean(scores), " | ".join(feedbacks)
_BASELINES["campaign_policy"] = _baseline_campaign_policy

def _baseline_agentic_trace_policy(self, signal):
    return "tools: note\nhint: observe the feedback"

AGENTIC_TRACE_CASES = [
    {"signal": "Need prior failures and family examples before proposing a prompt update.", "required": {"trace_search", "note"}, "hint_terms": {"prior", "failure", "family"}},
    {"signal": "Need validate a candidate on a small subset before accepting it.", "required": {"run_subset"}, "hint_terms": {"validate", "subset", "accept"}},
    {"signal": "Need inspect the saved artifact for syntax and current_code reuse.", "required": {"artifact_linter"}, "hint_terms": {"syntax", "artifact", "code"}},
    {"signal": "Task is saturated at 1.0 with zero gain; treat as control and avoid expensive tool calls.", "required": set(), "hint_terms": {"satur", "control", "avoid", "stop"}},
    {"signal": "Noisy transfer result: compare cold versus warm prior on held-out families before promoting.", "required": {"trace_search", "run_subset"}, "hint_terms": {"transfer", "holdout", "warm", "cold", "promot"}},
]

def evaluate_agentic_trace_policy(component, _task_id):
    scores, feedbacks = [], []
    for case in AGENTIC_TRACE_CASES:
        raw = component(case["signal"])
        text = _policy_text(raw)
        selected = parse_optimizer_tool_policy(raw, OPTIMIZER_TOOL_NAMES, max_tools=3)
        selected_set = set(selected)
        required = set(case["required"])
        expensive = selected_set - {"note"}
        if required:
            coverage = len(selected_set & required) / len(required)
            extras = len(selected_set - required - {"note"})
            tool_score = max(0.0, coverage - 0.15 * extras)
        else:
            tool_score = 1.0 if not expensive else max(0.0, 1.0 - 0.45 * len(expensive))
        hint_score = min(1.0, sum(1 for term in case["hint_terms"] if term.lower() in text) / 2.0)
        score = 0.70 * tool_score + 0.30 * hint_score
        scores.append(score)
        feedbacks.append(f"selected={selected}; required={sorted(required)}; score={score:.2f}")
    return statistics.mean(scores), " | ".join(feedbacks)
_BASELINES["agentic_trace_policy"] = _baseline_agentic_trace_policy

def _baseline_promotion_policy(self, artifact_report):
    return "action: promote\nreason: best score"

PROMOTION_CONFIDENCE_GATE = ConfidenceGate(min_support=2, z=1.0, min_gain=0.0)
PROMOTION_CASES = [
    {"name": "validated_code_gain", "report": {"kind": "code", "mean_score": 1.0, "initial": 0.625, "std": 0.0, "n": 2, "artifact_version": 1, "syntax_ok": True, "saturated_control": False}, "actions": {"promote"}, "required": {"code", "validated", "gain"}, "forbidden": {"reject", "control"}},
    {"name": "saturated_stride_control", "report": {"kind": "code", "mean_score": 1.0, "initial": 1.0, "std": 0.0, "n": 2, "artifact_version": 0, "syntax_ok": True, "saturated_control": True}, "actions": {"control", "archive", "do_not_promote", "skip"}, "required": {"satur", "control"}, "forbidden": {"promote"}},
    {"name": "single_seed_noisy_config", "report": {"kind": "config", "mean_score": 0.16, "initial": 0.13, "std": 0.08, "n": 1, "artifact_version": 0, "syntax_ok": True, "saturated_control": False}, "actions": {"retest", "hold", "probe"}, "required": {"config", "single", "retest"}, "forbidden": {"promote"}},
    {"name": "invalid_syntax_code", "report": {"kind": "code", "mean_score": -1.0, "initial": 0.4, "std": 0.0, "n": 2, "artifact_version": 1, "syntax_ok": False, "saturated_control": False}, "actions": {"reject", "repair"}, "required": {"syntax", "reject"}, "forbidden": {"promote"}},
    {"name": "warm_prior_regression", "report": {"kind": "prior", "mean_score": -0.09, "initial": -0.01, "std": 0.002, "n": 2, "artifact_version": 0, "syntax_ok": True, "saturated_control": False}, "actions": {"reject", "rollback", "cold", "do_not_promote"}, "required": {"regress", "rollback"}, "forbidden": {"promote"}},
]

def _promotion_guard_case(case):
    report = case["report"]
    required = set(case["required"])
    if PROMOTION_CONFIDENCE_GATE.needs_retest(report):
        required.update({"single", "retest"})
    elif not PROMOTION_CONFIDENCE_GATE.clears_promotion(report) and report.get("kind") in {"prior", "config"}:
        required.update({"regress", "rollback"})
    return GuardedDecisionCase(name=case["name"], payload=report, allowed_actions=tuple(sorted(case["actions"])), required_terms=tuple(sorted(required)), forbidden_terms=tuple(sorted(case["forbidden"])), weights={"action": 0.45, "required": 0.35, "forbidden": 0.20}, required_denominator=min(2, len(required)), hard_forbidden="promote" in case["forbidden"], forbidden_floor=0.0)

PROMOTION_GUARDED_EVALUATOR = GuardedDecisionEvaluator(tuple(_promotion_guard_case(case) for case in PROMOTION_CASES))
def evaluate_promotion_policy(component, _task_id):
    return PROMOTION_GUARDED_EVALUATOR(component, _task_id)
_BASELINES["promotion_policy"] = _baseline_promotion_policy

def _qasper_prompt_emitter(self):
    return ""
_BASELINES["qasper_prompt_emitter"] = _qasper_prompt_emitter


## UC1 — Optimize & validate a new Trace component (code surface)

Status in this notebook:

- **definitions** are centralized above,
- **execution** is explicit below,
- BBEH direct solver uses a real direct-answer evaluator,
- trace summarizer default/strict are kept as separate experiment rows.


In [ ]:

if enabled("uc1"):
    uc1 = [
        ("batch_design helper", run_code_experiment(
            "batch_design", "internal:batch_design",
            "Select the hard/failing items before easy ones; maximize validator score.",
            seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc1_batch_design", baseline=_weak_batch,
        )),
        ("trace_summarizer default", run_code_experiment(
            "trace_summarizer", "internal:code_param",
            "Preserve failing-assertion evidence while removing noise; be concise.",
            seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc1_trace_summarizer_default", baseline=_trunc_summary,
        )),
        ("trace_summarizer strict", run_code_experiment(
            "trace_summarizer", "internal:code_param",
            "Keep ALL error evidence, drop everything else, target <60 chars.",
            seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc1_trace_summarizer_strict", baseline=_trunc_summary,
        )),
        ("BBEH direct solver", run_code_experiment(
            "bbeh_direct_solver", "internal:multiobjective_bbeh",
            "Rewrite the Python function to parse BBEH boolean expressions. Input `question` ends with ' is'. Return exactly 'True' or 'False'.",
            seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc1_bbeh_direct_solver",
            baseline=_bbeh_direct_solver,
            evaluate=make_tracebench_direct_answer_evaluator("internal:multiobjective_bbeh", max_examples=MAX_EXAMPLES, normalizer=_norm_bool_answer),
        )),
    ]
    show_table("Use Case 1 — code surface", uc1)
else:
    print("UC1 skipped.")


## UC2 — Learn the best setup / default prompt for a family (config surface)

This refactor makes UC2 execution explicit. In the current notebook, `config_spec(...)`
exists as a generic reusable builder and is reused later by other sections; here the actual UC2
rows are materialized immediately below.


In [ ]:

if enabled("uc2"):
    KNOWLEDGE_MENU = ["", "Prefer verification before final answer.", "Use prior failures and examples before final answer."]
    uc2 = []
    uc2.append(("GSM8K artifact menu", run_spec_seeds(
        config_spec(["starting_artifact"], memory_root="./mem_uc2_gsm8k", task=FAMILY_TASK, family_name="reasoning"),
        seeds=DIAGNOSTIC_SEEDS, level_id="o1_setup", run_name="mem_uc2_gsm8k",
    )))
    uc2.append(("GSM8K + initial_knowledge / warm prior", run_spec_seeds(
        config_spec(["starting_artifact", "initial_knowledge"], reuse=True, extra_constraints={"initial_knowledge": KNOWLEDGE_MENU}, memory_root="./mem_uc2_gsm8k_warm_knowledge", task=FAMILY_TASK, family_name="reasoning"),
        seeds=DIAGNOSTIC_SEEDS, level_id="o1_setup", run_name="mem_uc2_gsm8k_warm_knowledge",
    )))
    drop_row = run_spec_seeds(
        config_spec(["starting_artifact"], memory_root="./mem_uc2_drop", task=HARD_PROMPT_TASKS["drop"], family_name="drop", max_examples=HARD_MAX_EXAMPLES),
        seeds=DIAGNOSTIC_SEEDS, level_id="o1_setup", run_name="mem_uc2_drop",
    )
    uc2.append(("DROP prompt config", mark_control(drop_row, "often saturated control")))
    uc2.append(("QASPER prompt config", run_spec_seeds(
        config_spec(["starting_artifact"], memory_root="./mem_uc2_qasper", task=HARD_PROMPT_TASKS["qasper"], family_name="qasper", max_examples=HARD_MAX_EXAMPLES),
        seeds=DIAGNOSTIC_SEEDS, level_id="o1_setup", run_name="mem_uc2_qasper",
    )))
    uc2.append(("mixed GSM8K+QASPER prompt config", run_spec_seeds(
        config_spec(["starting_artifact"], memory_root="./mem_uc2_mixed_gsm8k_qasper", task=FAMILY_TASK, tasks=[FAMILY_TASK, HARD_PROMPT_TASKS["qasper"]], family_name="mixed", max_examples=HARD_MAX_EXAMPLES),
        seeds=DIAGNOSTIC_SEEDS, level_id="o1_setup", run_name="mem_uc2_mixed_gsm8k_qasper",
    )))
    uc2.append(("QASPER causal numeric config", run_spec_seeds(
        config_spec(CAUSAL_NUMERIC_TARGETS, numeric_constraints=CAUSAL_NUMERIC_CONSTRAINTS, memory_root="./mem_uc2_qasper_numeric", task=HARD_PROMPT_TASKS["qasper"], family_name="qasper_numeric", max_examples=HARD_MAX_EXAMPLES, inner_steps=2),
        seeds=DIAGNOSTIC_SEEDS, level_id="o1_setup", run_name="mem_uc2_qasper_numeric",
    )))
    uc2.append(("QASPER Optuna numeric level", numeric_optimizer_arm(
        HARD_PROMPT_TASKS["qasper"], CAUSAL_NUMERIC_TARGETS, CAUSAL_NUMERIC_CONSTRAINTS,
        family_name="qasper_numeric", memory_root="./mem_uc2_qasper_optuna", max_examples=HARD_MAX_EXAMPLES, inner_steps=2,
    )))
    show_table("Use Case 2 — config surface", uc2)
else:
    print("UC2 skipped.")


## UC3 — Discover a new capability from a spec + objectives


In [ ]:

if enabled("uc3"):
    uc3 = [
        ("seed: weak (constant-answer headroom)", run_spec_seeds(
            capability_spec("Always answer 0. Do not plan, verify, decompose, or explain.", "./mem_uc3_weak"),
            seeds=SEEDS, level_id="cap", run_name="mem_uc3_weak",
        )),
        ("seed: terse", run_spec_seeds(
            capability_spec("Solve correctly using the fewest words.", "./mem_uc3_terse"),
            seeds=SEEDS, level_id="cap", run_name="mem_uc3_terse",
        )),
        ("seed: verify", run_spec_seeds(
            capability_spec("Make a short plan; solve; then verify/check the answer before replying.", "./mem_uc3_verify"),
            seeds=SEEDS, level_id="cap", run_name="mem_uc3_verify",
        )),
        ("seed: decompose", run_spec_seeds(
            capability_spec("Plan, decompose into sub-steps, solve each, then verify before answering.", "./mem_uc3_decompose"),
            seeds=SEEDS, level_id="cap", run_name="mem_uc3_decompose",
        )),
    ]
    show_table("Use Case 3 — capability discovery", uc3)
else:
    print("UC3 skipped.")


## UC4 — Family policy (O2) & transferable prior (O3)


In [ ]:

if enabled("uc4"):
    uc4 = []
    uc4.append(("O2 family policy", run_spec_seeds(
        family_policy_spec("o2", warm=False, memory_root="./mem_uc4_o2_policy"),
        seeds=DIAGNOSTIC_SEEDS, level_id="o2_policy", run_name="mem_uc4_o2_policy",
    )))
    uc4.append(("O2 causal numeric policy", run_spec_seeds(
        family_policy_spec("o2", warm=False, targets=CAUSAL_NUMERIC_TARGETS, constraints=CAUSAL_NUMERIC_CONSTRAINTS, inner_steps=2, memory_root="./mem_uc4_o2_numeric"),
        seeds=DIAGNOSTIC_SEEDS, level_id="o2_policy", run_name="mem_uc4_o2_numeric",
    )))
    uc4.append(("O2 Optuna numeric level", numeric_optimizer_arm(
        FAMILY_TASK, CAUSAL_NUMERIC_TARGETS, CAUSAL_NUMERIC_CONSTRAINTS,
        tasks=[FAMILY_TASK, HARD_PROMPT_TASKS["qasper"]], family_name="uc4_o2",
        memory_root="./mem_uc4_o2_optuna", max_examples=HARD_MAX_EXAMPLES, inner_steps=2,
    )))
    uc4.append(("O3 cold prior", run_spec_seeds(
        family_policy_spec("o3", warm=False, memory_root="./mem_uc4_o3_cold"),
        seeds=DIAGNOSTIC_SEEDS, level_id="o3_prior", run_name="mem_uc4_o3_cold",
    )))
    uc4.append(("O3 warm prior", run_spec_seeds(
        family_policy_spec("o3", warm=True, memory_root="./mem_uc4_o3_warm"),
        seeds=DIAGNOSTIC_SEEDS, level_id="o3_prior", run_name="mem_uc4_o3_warm",
    )))
    show_table("Use Case 4 — family policy & transferable prior", uc4)
else:
    print("UC4 skipped.")


## UC5 — Code helpers vs optimizer-side tools


In [ ]:

if enabled("uc5"):
    uc5 = []
    uc5.append(("code helper: take_last", run_code_experiment(
        "batch_design", "internal:batch_design",
        "Select hard/failing items first to maximize validator score.",
        seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc5_code_take_last", baseline=_baseline_take_last,
    )))
    stride_row = run_code_experiment(
        "batch_design", "internal:batch_design",
        "Verify that saturated helper baselines are detected as no-op controls.",
        seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc5_code_stride", baseline=_baseline_stride,
    )
    uc5.append(("code helper: stride", mark_control(stride_row, "saturated helper control")))
    uc5.append(("optimizer tool policy", run_code_experiment(
        "optimizer_tool_policy", "internal:optimizer_tool_policy",
        "Return a compact tools: ... policy selecting only optimizer tools needed by the signal; avoid expensive tools on saturated controls.",
        seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc5_tool_policy", baseline=_baseline_tool_policy, evaluate=evaluate_optimizer_tool_policy,
    )))
    show_table("Use Case 5 — code helpers vs optimizer-side tools", uc5)
else:
    print("UC5 skipped.")


## UC6 — Which feedback channel helps the optimizer?


In [ ]:

if enabled("uc6"):
    uc6 = [
        (
            f"trace_type={tt} | credit_horizon=step",
            run_spec_seeds(feedback_spec(f"o1_trace_{tt}", tt), seeds=DIAGNOSTIC_SEEDS, level_id=f"o1_trace_{tt}", run_name=f"mem_uc6_trace_{tt}")
        )
        for tt in ["internal", "otel", "hybrid"]
    ]
    uc6.append(("trace_type=internal | causal numeric config (inner_steps=2)", run_spec_seeds(
        feedback_spec("o1_trace_internal_numeric", "internal", targets=CAUSAL_NUMERIC_TARGETS, constraints=CAUSAL_NUMERIC_CONSTRAINTS, inner_steps=2, memory_root="./mem_uc6_trace_internal_numeric"),
        seeds=DIAGNOSTIC_SEEDS, level_id="o1_trace_internal_numeric", run_name="mem_uc6_trace_internal_numeric",
    )))
    uc6.append(("trace_type=internal numeric-optimizer (Optuna, inner_steps=2)", numeric_optimizer_arm(
        UC6_TASK, CAUSAL_NUMERIC_TARGETS, CAUSAL_NUMERIC_CONSTRAINTS,
        family_name="uc6_internal_numeric", memory_root="./mem_uc6_internal_numeric", max_examples=HARD_MAX_EXAMPLES, inner_steps=2,
    )))
    show_table("Use Case 6 — trace representation diagnostic", uc6)
else:
    print("UC6 skipped.")


## UC7 — Graph routing to a sub-optimizer tool


In [ ]:

if enabled("uc7"):
    from argparse import Namespace
    try:
        from examples.recursive_opt_abc_probe import run_suboptimizer_graph, run_conditional_suboptimizer_graph
        _UC7_ENABLED = True
        _UC7_IMPORT_ERROR = None
    except Exception as exc:
        run_suboptimizer_graph = None
        run_conditional_suboptimizer_graph = None
        _UC7_ENABLED = False
        _UC7_IMPORT_ERROR = str(exc)

    def run_suboptimizer_use_case(runner, artifact_id, reason):
        if not LIVE:
            return {"scores": [], "initial": None, "wall_s": None, "artifact": "(offline preflight: set LIVE=True to optimize graph route)", "artifact_id": None, "artifact_file": None, "spec_file": None, "dry": True}
        if not _UC7_ENABLED or runner is None:
            return {"scores": [], "initial": None, "wall_s": None, "artifact": "(skipped: missing langgraph/probe dependencies)", "artifact_id": None, "artifact_file": None, "spec_file": None, "errors": [f"UC7 unavailable: {_UC7_IMPORT_ERROR}"], "dry": False}
        reset_standard_budget()
        args = Namespace(model=MODEL, iterations=RUN_ITERATIONS, candidates=NUM_CANDIDATES, max_examples=MAX_EXAMPLES, timeout_seconds=TIMEOUT_S, live=True, skip_preflight=True)
        result = runner(OUTPUT_ROOT, args)
        artifact = json.dumps({"params": result.get("params"), "score_history": result.get("score_history"), "oracle_tool_score": result.get("oracle_tool_score"), "always_tool_score": result.get("always_tool_score")}, indent=2, sort_keys=True)
        return {"scores": [float(result["final"])], "initial": float(result["initial"]), "wall_s": float(result["wall_s"]), "artifact": artifact, "artifact_id": artifact_id, "artifact_file": result.get("artifact_file"), "spec_file": result.get("spec_file"), "errors": [], "control_reason": reason}

    uc7 = [
        ("graph route: always-use SciPy suboptimizer", run_suboptimizer_use_case(run_suboptimizer_graph, "graph:suboptimizer:latest", "learned graph route to SciPy sub-optimizer")),
        ("graph route: conditional cost-aware suboptimizer", run_suboptimizer_use_case(run_conditional_suboptimizer_graph, "graph:conditional_suboptimizer:latest", "tests conditional routing under tool cost")),
    ]
    show_table("Use Case 7 — graph/suboptimizer routing", uc7)
else:
    print("UC7 skipped.")


## UC8 — Meta-campaign policy


In [ ]:

if enabled("uc8"):
    uc8 = [("adaptive campaign policy", run_code_experiment(
        "campaign_policy", "internal:campaign_policy",
        "Rewrite a compact if/elif campaign controller. Return action, task(s), max_examples, and reason. Strict requirements: saturated runs must stop/control with max_examples <=2 and avoid the saturated task; high-spread BBEH should exploit/train BBEH with 8-16 examples; QASPER should probe with 3-6 examples; low-spread stalled GSM8K should switch/restart toward QASPER or BBEH with 3-8 examples; mixed regressions should split/ablate into QASPER plus BBEH. Include reason words matching the decision.",
        seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc8_campaign_policy",
        baseline=_baseline_campaign_policy, evaluate=evaluate_campaign_policy,
    ))]
    show_table("Use Case 8 — meta-campaign policy", uc8)
else:
    print("UC8 skipped.")


## UC9 — Agentic Trace policy: tools + hints


In [ ]:

if enabled("uc9"):
    uc9 = [("tool+hint policy", run_code_experiment(
        "agentic_trace_policy", "internal:agentic_trace_policy",
        "Rewrite a compact policy function. Given a signal string, return tools: ... and hint: ... . Select only useful optimizer-side tools. Avoid expensive tools on saturated controls; use trace_search/run_subset for noisy transfer; use artifact_linter for code/syntax reuse.",
        seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc9_agentic_trace_policy",
        baseline=_baseline_agentic_trace_policy, evaluate=evaluate_agentic_trace_policy,
    ))]
    show_table("Use Case 9 — agentic trace policy", uc9)
else:
    print("UC9 skipped.")


## UC10 — Artifact promotion policy


In [ ]:

if enabled("uc10"):
    uc10 = [("artifact promotion policy", run_code_experiment(
        "promotion_policy", "internal:artifact_promotion_policy",
        "Rewrite a compact artifact gate. Given an artifact_report dict, return action and reason. Promote only validated code with real gain; archive saturated controls; retest single-seed/noisy config artifacts; reject or repair syntax failures; rollback warm-prior regressions. Mention the artifact kind and evidence in the reason.",
        seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc10_promotion_policy",
        baseline=_baseline_promotion_policy, evaluate=evaluate_promotion_policy,
    ))]
    show_table("Use Case 10 — artifact promotion policy", uc10)
else:
    print("UC10 skipped.")


## UC11 — Code-emitted Trace-Bench prompt artifact


In [ ]:

if enabled("uc11"):
    uc11 = [("QASPER prompt-emitter code", run_code_experiment(
        "qasper_prompt_emitter", "hf:qasper",
        "Rewrite the function so it returns a concise QASPER starting_artifact prompt. The prompt should improve evidence-grounded QA: read the passage, identify supporting evidence, answer briefly, and avoid hallucinating when evidence is missing. Return only the prompt string; do not call the model or dataset inside this function.",
        seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc11_qasper_prompt_emitter",
        baseline=_qasper_prompt_emitter,
        evaluate=make_artifact_emitter_evaluator("hf:qasper", max_examples=HARD_MAX_EXAMPLES, credit_horizon="step"),
    ))]
    show_table("Use Case 11 — prompt emitter", uc11)
else:
    print("UC11 skipped.")


In [ ]:

# ====================== UC12 — promoted primitives ===========================
if enabled("uc12"):
    from opto.optimizers.optimizer import Optimizer
    from opto.trace.nodes import node as trace_node
    from opto.features.recursive_opt import (
        run_spec as recursive_run_spec,
        RepeatedResult,
        seed_everything,
        route_optimizers,
        OptunaOptimizer,
        LeastSquaresOptimizer,
        field_search_space,
        run_search_policy,
        make_search_policy_tool,
        make_search_policy_evaluator,
    )
    from opto.features.recursive_opt.budget import (
        make_budget as make_recursive_budget,
        budget_to_spec_dict,
        current_budget,
        reset_budget,
    )
    from opto.features.recursive_opt import spec as recursive_spec
    from opto.features.recursive_opt import tracebench as TB
    from opto.features.recursive_opt.effects import effects_for
    from opto.features.recursive_opt.levels import LevelConfig
    from opto.features.recursive_opt.tracebench import _summarize_feedbacks

    UC12_ROWS = []

    class NotebookNoLLMOptimizer(Optimizer):
        def __init__(self, parameters, **kwargs):
            super().__init__(parameters)
            self.steps = 0
        def step(self, *args, **kwargs):
            self.steps += 1
            return {}
        def zero_feedback(self):
            return None
        def backward(self, *args, **kwargs):
            return None

    def record_uc12(item, variant, status, metric=None, artifact=None, notes=""):
        UC12_ROWS.append({"item": item, "variant": variant, "status": status, "metric": metric, "artifact": artifact, "notes": notes})

    def uc12_table(rows):
        head = "| item | variant | status | metric | artifact/result | notes |\n|---|---|---|---:|---|---|"
        lines = [head]
        for row in rows:
            metric = row.get("metric")
            metric_txt = _fmt(metric) if isinstance(metric, (int, float)) else metric
            lines.append(f"| {_md_cell(row['item'])} | {_md_cell(row['variant'])} | {_md_cell(row['status'])} | {_md_cell(metric_txt)} | {(_md_code(row.get('artifact')) if row.get('artifact') else '-')} | {_md_cell(row.get('notes'))} |")
        return "\n".join(lines)

    def deterministic_capability_eval(capability_callable, _family):
        text = str(capability_callable(task="uc12").get("answer", ""))
        score = 1.0 if text else 0.0
        return {"accuracy": score}, f"deterministic capability score={score}", score

    budget_dict = {"wall_time_s": 1500, "optimizer_llm_calls": 8, "eval_llm_calls": 24, "candidates": 16, "on_exceed": "raise"}
    roundtrip_ok = budget_to_spec_dict(make_recursive_budget(budget_dict)) == budget_dict
    record_uc12("Item 4 budget", "lossless make_budget/to_spec_dict", "pass" if roundtrip_ok else "fail", metric=1.0 if roundtrip_ok else 0.0, notes="Dict, object, and method forms share one mapping.")

    spec_with_budget = {"memory_root": memory_path("mem_uc12_budget_override"), "budget": {"candidates": 99}, "levels": [make_level_spec(id="budget_probe", surface="capability", seed="probe", evaluator=deterministic_capability_eval, iterations=1)]}
    before_budget = dict(spec_with_budget["budget"])
    reset_budget()
    out_budget = recursive_run_spec(spec_with_budget, optimizer=NotebookNoLLMOptimizer, budget={"candidates": 7, "on_exceed": "return_best"})
    override_ok = current_budget().max_candidates == 7 and spec_with_budget["budget"] == before_budget
    record_uc12("Item 4 budget", "run_spec budget override isolation", "pass" if override_ok else "fail", metric=current_budget().max_candidates, artifact=out_budget["results"]["budget_probe"].get("artifact_id"), notes="Override applies to the run and leaves spec['budget'] unmutated.")

    import random
    seed_everything(123)
    first_random = [round(random.random(), 6) for _ in range(3)]
    seed_everything(123)
    second_random = [round(random.random(), 6) for _ in range(3)]
    record_uc12("Item 3 seeds", "seed_everything controls RNG", "pass" if first_random == second_random else "fail", metric=1.0 if first_random == second_random else 0.0, notes=f"random sequence={first_random}")

    seed_spec = {"memory_root": memory_path("mem_uc12_seeded"), "budget": {"candidates": 20, "on_exceed": "return_best"}, "levels": [make_level_spec(id="seeded", surface="capability", seed="seeded policy", evaluator=deterministic_capability_eval, iterations=1)]}
    seeded = recursive_run_spec(seed_spec, seeds=[0, 1, 2], optimizer=NotebookNoLLMOptimizer)
    seed_rr = seeded["seeded"]
    record_uc12("Item 3 seeds", "run_spec(seeds=) returns RepeatedResult", "pass" if isinstance(seed_rr, RepeatedResult) else "fail", metric=seed_rr.mean(), artifact=memory_path("mem_uc12_seeded_seed0"), notes=f"n_valid={seed_rr.n_valid()}, errors={len(seed_rr.errors)}")

    try:
        active_adapter = TB.TraceBenchTaskAdapter.from_config(tracebench_block(max_examples=min(6, MAX_EXAMPLES), inner_steps=1, timeout_seconds=25, eval_kwargs={"n_train": 6, "n_val": 1}))
        TB.register_task_adapter(active_adapter)
        contract = effects_for(active_adapter)
        active_ok = bool(contract["batch_design"].active and contract["credit_horizon"].active)
        record_uc12("Item 1 active fields", "adapter effect contract", "pass" if active_ok else "fail", metric=1.0 if active_ok else 0.0, notes=f"batch_design={contract['batch_design'].effects}; credit_horizon={contract['credit_horizon'].effects}")

        task_id = "internal:multiobjective_bbeh"
        bundle = active_adapter._load_bundle(task_id, fresh=True)
        train_dataset = bundle["train_dataset"]
        inputs = list(train_dataset.get("inputs") or [])[: min(6, active_adapter.max_examples)]
        infos = list(train_dataset.get("infos") or train_dataset.get("info") or [None] * len(inputs))[: len(inputs)]
        batch_orders = {}
        for design in ["random", "failure_balanced", "curriculum", "diversity"]:
            ordered_inputs, _ordered_infos = active_adapter._order_by_batch_design(inputs, infos, LevelConfig(batch_design=design, batch_size=4))
            batch_orders[design] = [len(str(value)) for value in ordered_inputs]
        order_changed = len({tuple(order) for order in batch_orders.values()}) > 1
        record_uc12("Item 1 active fields", "batch_design changes inner-training batch order", "pass" if order_changed else "flat", metric=len({tuple(order) for order in batch_orders.values()}), notes=f"orders_by_input_length={batch_orders}")
    except Exception as exc:
        record_uc12("Item 1 active fields", "real Trace-Bench batch_design probe", "fail", notes=_one_line_error(exc))

    feedbacks = [f"feedback {i}: failure mode {i % 3}" for i in range(5)]
    horizon_lengths = {h: len(_summarize_feedbacks(feedbacks, h)) for h in ["truncated", "episode", "step", "full"]}
    horizon_ok = len(set(horizon_lengths.values())) > 1
    record_uc12("Item 1 active fields", "credit_horizon changes optimizer-visible feedback", "pass" if horizon_ok else "fail", metric=max(horizon_lengths.values()) - min(horizon_lengths.values()), notes=f"summary lengths={horizon_lengths}")

    plan = route_optimizers(["starting_artifact", "batch_design", "batch_size"], policy={"order": "numeric_then_text", "numeric_optimizer": "optuna"})
    routing_ok = plan["numeric_fields"] == ["batch_design", "batch_size"] and plan["text_fields"] == ["starting_artifact"]
    record_uc12("Item 2 numeric routing", "mixed target routing", "pass" if routing_ok else "fail", metric=len(plan["numeric_fields"]), notes=str(plan))

    def numeric_eval(assignment):
        score = 0.6 if assignment.get("batch_design") == "failure_balanced" else 0.0
        score += 0.4 * (assignment.get("batch_size", 1) / 8.0)
        return score

    opt = OptunaOptimizer([trace_node("x", trainable=True, name="uc12_cfg")], evaluate=numeric_eval, space=field_search_space(["batch_design", "batch_size"]), max_trials=30)
    best_numeric = opt.step()
    best_numeric_score = max(score for _assignment, score in opt.history)
    record_uc12("Item 2 numeric routing", "OptunaOptimizer/fallback learns categorical+int optimum", "pass" if best_numeric == {"batch_design": "failure_balanced", "batch_size": 8} else "fail", metric=best_numeric_score, artifact=str(best_numeric), notes=f"history_len={len(opt.history)}")

    ls = LeastSquaresOptimizer([trace_node("x", trainable=True, name="uc12_ls")], evaluate=lambda a: a.get("batch_size", 1) / 8.0, space=field_search_space(["batch_size"]), max_trials=15, target=1.0)
    ls.step()
    record_uc12("Item 2 numeric routing", "LeastSquaresOptimizer handles integer numeric field", "pass" if ls.best_assignment and ls.best_assignment.get("batch_size") == 8 else "fail", metric=ls.best_assignment.get("batch_size") if ls.best_assignment else None, notes="Continuous solve rounded back into the integer field domain.")

    try:
        TB.register_task_adapter(TB.TraceBenchTaskAdapter.from_config(tracebench_block(max_examples=2, inner_steps=1, timeout_seconds=20)))
        families = {"combo": ["llm4ad:online_bin_packing_local"], "math": ["internal:multiobjective_gsm8k"]}
        mem_prior = MemoryLite(root=memory_path("mem_uc12_prior_defaults"))
        o2 = recursive_spec.compile_level(make_level_spec(id="o2_default", surface="family_policy", family="*"), mem_prior, families)
        o3 = recursive_spec.compile_level(make_level_spec(id="o3_default", surface="prior", family="*"), mem_prior, families)
        defaults_ok = ("starting_artifact" in o2._fields and "batch_design" in o3._fields)
        record_uc12("Item 5 priors", "active default policy/prior fields", "pass" if defaults_ok else "fail", metric=len(o2._fields), notes=f"o2_fields={o2._fields}; o3_fields={o3._fields}")

        doc_adapter = TB.TraceBenchTaskAdapter.from_config(tracebench_block(max_examples=1, inner_steps=0, timeout_seconds=10))
        doc_node = trace_node("clean artifact", trainable=True, name="uc12_artifact")
        doc_adapter._apply_starting_artifact({"param": doc_node}, LevelConfig(initial_knowledge="Prefer verification before final answer."))
        description = "\n".join(str(value) for value in [getattr(doc_node, "description", ""), getattr(doc_node, "_description", "")] if value)
        docs_ok = "prefer verification" in description.lower() and str(doc_node.data) == "clean artifact"
        record_uc12("Item 5 priors", "initial_knowledge reaches optimizer docs", "pass" if docs_ok else "fail", metric=1.0 if docs_ok else 0.0, notes="Artifact text stays clean; family prior is in the trainable node description.")
    except Exception as exc:
        record_uc12("Item 5 priors", "prior/default validation", "fail", notes=_one_line_error(exc))

    mem_search = MemoryLite(root=memory_path("mem_uc12_search_policy"))
    for i, fb in enumerate(["parse failures disappear when examples include expected output format", "timeouts improve after preferring short candidate programs", "arithmetic answers need final verification", "parse failures recur when prompt omits JSON schema"]):
        mem_search.record(level="O1", cfg={"i": i}, family="codegen", score=0.1 * i, feedback=fb)

    recent_lesson = run_search_policy({"k": 2, "strategy": "recent", "template": "Lesson: {lessons}"}, mem_search, family="codegen")
    diverse_lesson = run_search_policy({"k": 2, "strategy": "diverse", "template": "Lesson: {lessons}"}, mem_search, family="codegen")
    record_uc12("Item 6 search policy", "policy changes retrieved lesson", "pass" if recent_lesson != diverse_lesson else "flat", metric=abs(len(recent_lesson) - len(diverse_lesson)), notes=f"recent='{recent_lesson[:60]}'; diverse='{diverse_lesson[:60]}'")

    tool = make_search_policy_tool({"k": 3, "strategy": "all", "template": "Avoid: {lessons}"}, mem_search, family="codegen")
    tool_text = tool("")
    record_uc12("Item 6 search policy", "search policy tool emits lesson text", "pass" if "avoid:" in tool_text.lower() else "fail", metric=len(tool_text), notes=tool_text[:80])

    evaluator = make_search_policy_evaluator(mem_search, lambda prior: 0.4 + (0.4 if "parse" in prior.lower() else 0.0), family="codegen")
    lift, feedback = evaluator({"k": 2, "strategy": "recent", "template": "Avoid: {lessons}"})
    record_uc12("Item 6 search policy", "search policy evaluator shows positive lift", "pass" if lift > 0 else "fail", metric=lift, notes=feedback)

    display(Markdown("### Use Case 12 — six promoted recursive_opt primitives\n" + uc12_table(UC12_ROWS)))
    uc12 = [("promoted primitives", {
        "scores": [1.0 if row["status"] == "pass" else 0.0 for row in UC12_ROWS],
        "initial": None, "wall_s": None,
        "artifact": json.dumps(UC12_ROWS, indent=2, sort_keys=True),
        "artifact_id": None,
        "artifact_file": write_experiment_json(memory_path("mem_uc12"), "uc12_six_promotions.json", UC12_ROWS),
        "best_step": None, "artifact_version": None, "progress": None, "spec_file": None,
        "errors": [], "dry": False, "notes": "engineering validation only",
    })]
else:
    print("UC12 skipped.")


In [ ]:

# =========================== UC13 — numeric head-to-head ====================
if enabled("uc13"):
    from opto.features.recursive_opt import optimize_config_numeric
    from opto.features.recursive_opt import spec as recursive_spec
    from opto.features.recursive_opt import tracebench as TB
    from opto.features.recursive_opt.levels import LevelConfig, MetaLevel

    UC13_TASK_OVERRIDE = os.environ.get("RECURSIVE_OPT_UC13_TASK")
    UC13_CANDIDATE_TASKS = [task.strip() for task in os.environ.get("RECURSIVE_OPT_UC13_TASK_CANDIDATES", "hf:qasper,internal:multiobjective_gsm8k").split(",") if task.strip()]
    UC13_TASK = UC13_TASK_OVERRIDE or (UC13_CANDIDATE_TASKS[0] if UC13_CANDIDATE_TASKS else "hf:qasper")
    UC13_FIELDS = list(CAUSAL_NUMERIC_TARGETS)
    UC13_MAX_EXAMPLES = int(os.environ.get("RECURSIVE_OPT_UC13_MAX_EXAMPLES", "6"))
    UC13_PILOT_MAX_EXAMPLES = int(os.environ.get("RECURSIVE_OPT_UC13_PILOT_MAX_EXAMPLES", str(min(4, UC13_MAX_EXAMPLES))))
    UC13_INNER_STEPS = int(os.environ.get("RECURSIVE_OPT_UC13_INNER_STEPS", "2"))
    UC13_TRIALS = int(os.environ.get("RECURSIVE_OPT_UC13_TRIALS", "12" if LIVE else "24"))
    UC13_OFFLINE_TRIALS = int(os.environ.get("RECURSIVE_OPT_UC13_OFFLINE_TRIALS", "24"))
    UC13_BUDGET = {**budget_block(), "optimizer_llm_calls": 8, "eval_llm_calls": min(MAX_EVAL_CALLS, 48), "candidates": min(MAX_CANDIDATES, 8)}
    UC13_PILOT_ASSIGNMENTS = [{"batch_design": "random", "batch_size": 2}, {"batch_design": "failure_balanced", "batch_size": 8}, {"batch_design": "diversity", "batch_size": 2}, {"batch_design": "curriculum", "batch_size": 4}]

    def uc13_tracebench_block(max_examples=None):
        return tracebench_block(max_examples=max_examples or UC13_MAX_EXAMPLES, inner_steps=UC13_INNER_STEPS)

    def _uc13_safe_name(task):
        return str(task).replace(":", "_").replace("/", "_").replace(".", "_")

    def _uc13_config_from_assignment(assignment):
        cfg = LevelConfig()
        for field, value in assignment.items():
            setattr(cfg, field, value)
        return cfg

    def _uc13_result(root, label, initial, best_assignment, best_score, history, wall_s, spec_file=None, notes=""):
        eval_calls = len(history)
        curve = [round(float(score), 3) for _assignment, score in history]
        payload = {"label": label, "initial": initial, "best_assignment": best_assignment, "best_score": best_score, "history": history, "curve": curve, "task": UC13_TASK, "fields": UC13_FIELDS, "live": LIVE, "wall_s": round(float(wall_s), 1), "eval_calls": eval_calls}
        artifact_file = write_experiment_json(root, "uc13_numeric_result.json", payload)
        artifact = json.dumps({"best_assignment": best_assignment, "best_score": best_score, "curve": curve, "history_len": eval_calls, "task": UC13_TASK}, indent=2, sort_keys=True, default=str)
        return {"scores": [float(best_score)], "initial": float(initial), "wall_s": round(float(wall_s), 1), "eval_calls": eval_calls, "artifact": artifact, "artifact_id": "uc13:numeric:best", "artifact_file": artifact_file, "best_step": (max(range(len(curve)), key=lambda index: curve[index]) if curve else None), "artifact_version": None, "progress": {"history": history}, "spec_file": spec_file, "errors": [], "dry": False, "notes": notes or "zero LLM proposal calls; each trial still runs the real inner evaluator"}

    def run_uc13_offline_preflight():
        root = memory_path("mem_uc13_offline_numeric")
        mem = MemoryLite(root=root)
        def offline_runner(cfg, _task):
            design_score = {"failure_balanced": 0.55, "diversity": 0.35, "curriculum": 0.20, "random": 0.05}.get(cfg.batch_design, 0.0)
            batch_score = min(max(float(cfg.batch_size), 1.0), 8.0) / 8.0 * 0.35
            score = design_score + batch_score + 0.10
            return score, f"causal_preflight design={cfg.batch_design} batch_size={cfg.batch_size} score={score:.3f}"
        level = MetaLevel(cfg=LevelConfig(), inner_runner=offline_runner, trainable_fields=tuple(UC13_FIELDS), memory=mem)
        initial, _ = offline_runner(LevelConfig(), "offline_causal_preflight")
        t0 = time.time()
        best, best_score, history = optimize_config_numeric(level, "offline_causal_preflight", UC13_FIELDS, max_trials=UC13_OFFLINE_TRIALS, space=numeric_search_space(UC13_FIELDS, CAUSAL_NUMERIC_CONSTRAINTS))
        return _uc13_result(root, "offline numeric causal preflight", initial, best, best_score, history, time.time() - t0)

    def uc13_live_numeric_spec(root, task=None, max_examples=None):
        task_id = task or UC13_TASK
        return {"families": {"uc13": [task_id]}, "memory_root": root, "budget": dict(UC13_BUDGET), "tracebench": uc13_tracebench_block(max_examples=max_examples), "scoring": {"clip": [-1.0, 1.0]}, "levels": [make_level_spec(id="uc13_numeric", surface="config", family="uc13", task=task_id, targets=UC13_FIELDS, constraints=CAUSAL_NUMERIC_CONSTRAINTS, fixed={"optimizer": TRACEBENCH_OPTIMIZER, "trainer": "PrioritySearch", "trace_type": "internal", "credit_horizon": "step"}, iterations=RUN_ITERATIONS)]}

    def run_uc13_task_signal_pilot():
        if not LIVE:
            return UC13_TASK, mark_control({"scores": [], "initial": None, "wall_s": None, "eval_calls": 0, "artifact": "(offline: UC13 task pilot skipped)", "artifact_id": None, "artifact_file": None, "spec_file": None, "errors": [], "dry": True}, "offline task pilot skipped")
        if UC13_TASK_OVERRIDE:
            return UC13_TASK_OVERRIDE, mark_control({"scores": [0.0], "initial": 0.0, "wall_s": 0.0, "eval_calls": 0, "artifact": f"task override: {UC13_TASK_OVERRIDE}", "artifact_id": None, "artifact_file": None, "spec_file": None, "errors": [], "dry": False}, "task override supplied; spread pilot skipped")
        root = Path(memory_path("mem_uc13_task_signal_pilot"))
        started = time.time()
        task_rows, errors = [], []
        for task_id in UC13_CANDIDATE_TASKS:
            try:
                task_root = root / _uc13_safe_name(task_id)
                spec = uc13_live_numeric_spec(str(task_root), task=task_id, max_examples=UC13_PILOT_MAX_EXAMPLES)
                spec_file = write_experiment_json(task_root, "spec.json", spec)
                TB.configure_tracebench_adapter(spec["tracebench"], require=True)
                mem = MemoryLite(root=str(task_root))
                level = recursive_spec.compile_level(spec["levels"][0], mem, spec["families"], spec.get("scoring"))
                for assignment in UC13_PILOT_ASSIGNMENTS:
                    score, _feedback = level._inner_runner(_uc13_config_from_assignment(assignment), task_id)
                    task_rows.append({"task": task_id, "assignment": dict(assignment), "score": float(score), "spec_file": spec_file})
            except Exception as exc:
                errors.append(f"{task_id}: {_one_line_error(exc)}")
        summaries = []
        for task_id in UC13_CANDIDATE_TASKS:
            scores = [row["score"] for row in task_rows if row["task"] == task_id]
            if not scores:
                continue
            summaries.append({"task": task_id, "min": min(scores), "max": max(scores), "spread": max(scores) - min(scores), "mean": statistics.mean(scores), "n": len(scores)})
        selected = max(summaries, key=lambda row: (row["spread"], row["max"])) if summaries else {"task": UC13_TASK, "min": 0.0, "max": 0.0, "spread": 0.0, "mean": 0.0, "n": 0}
        payload = {"selected_task": selected["task"], "summaries": summaries, "rows": task_rows, "errors": errors, "assignments": UC13_PILOT_ASSIGNMENTS, "max_examples": UC13_PILOT_MAX_EXAMPLES}
        artifact_file = write_experiment_json(root, "uc13_task_signal_pilot.json", payload)
        result = {"scores": [float(selected["max"])], "initial": float(selected["min"]), "wall_s": round(time.time() - started, 1), "eval_calls": len(task_rows), "artifact": json.dumps(payload, indent=2, sort_keys=True, default=str), "artifact_id": "uc13:task_signal:pilot", "artifact_file": artifact_file, "best_step": None, "artifact_version": None, "progress": {"rows": task_rows}, "spec_file": None, "errors": errors, "dry": False, "notes": f"selected {selected['task']} by spread={selected['spread']:.3f}; task-selection pilot, not an optimizer result"}
        return selected["task"], mark_control(result, "task-selection pilot; not optimizer evidence")

    def run_uc13_live_numeric():
        root = memory_path("mem_uc13_live_numeric")
        spec = uc13_live_numeric_spec(root)
        spec_file = write_experiment_json(root, "spec.json", spec)
        TB.configure_tracebench_adapter(spec["tracebench"], require=True)
        mem = MemoryLite(root=root)
        level = recursive_spec.compile_level(spec["levels"][0], mem, spec["families"], spec.get("scoring"))
        initial, _ = level._inner_runner(LevelConfig(), UC13_TASK)
        reset_standard_budget()
        t0 = time.time()
        best, best_score, history = optimize_config_numeric(level, UC13_TASK, UC13_FIELDS, max_trials=UC13_TRIALS, space=numeric_search_space(UC13_FIELDS, CAUSAL_NUMERIC_CONSTRAINTS))
        return _uc13_result(root, "live numeric config search", initial, best, best_score, history, time.time() - t0, spec_file=spec_file, notes=f"selected_task={UC13_TASK}; zero LLM proposal calls; eval_trials={len(history)}")

    def uc13_live_llm_spec():
        return config_spec(UC13_FIELDS, numeric_constraints=CAUSAL_NUMERIC_CONSTRAINTS, memory_root="./mem_uc13_live_llm", task=UC13_TASK, family_name="uc13", max_examples=UC13_MAX_EXAMPLES, inner_steps=UC13_INNER_STEPS, budget=UC13_BUDGET)

    uc13 = [("offline numeric causal preflight", run_uc13_offline_preflight())]
    if LIVE:
        UC13_TASK, uc13_pilot = run_uc13_task_signal_pilot()
        uc13.append(("live task-signal pilot", uc13_pilot))
        uc13.append(("live numeric-only active config", run_uc13_live_numeric()))
        uc13.append(("live LLM active config", run_spec_seeds(uc13_live_llm_spec(), seeds=DIAGNOSTIC_SEEDS, level_id="o1_setup", run_name="mem_uc13_live_llm")))
    else:
        uc13.append(("live numeric-only active config", {"scores": [], "initial": None, "wall_s": None, "eval_calls": None, "artifact": "(offline preflight only: set LIVE=True for real Trace-Bench head-to-head)", "artifact_id": None, "artifact_file": None, "spec_file": None, "errors": [], "dry": True}))
    show_table("Use Case 13 — numeric config optimizer head-to-head", uc13)
else:
    print("UC13 skipped.")


In [ ]:

# =================== Three-way benchmark / Stage 2 scaffolding ==============
if enabled("three_way") or enabled("stage2") or enabled("uc14") or enabled("tier_followups"):
    from examples.recursive_opt_three_way import (
        benchmark_uc,
        benchmark_uc_budget_sweep,
        run_numeric_arm,
        make_code_arm,
        markdown_report,
        promotion_decision,
        iters_to_peak,
        numeric_landscape_evaluator,
        count_tool_calls,
        make_solver_critic_evaluator,
    )

    TW_TOTAL_CANDIDATES = int(os.environ.get("RECURSIVE_OPT_TW_CANDIDATES", max(18, RUN_ITERATIONS * NUM_CANDIDATES * 4)))
    TW_NUM_CANDIDATES = NUM_CANDIDATES
    TW_SEEDS = list(SEEDS) if len(SEEDS) >= 3 else [0, 1, 2]
    TW_OPTIMIZER_CALLS = int(os.environ.get("RECURSIVE_OPT_TW_OPT_CALLS", "12"))
    TW_EVAL_CALLS = int(os.environ.get("RECURSIVE_OPT_TW_EVAL_CALLS", "64"))
    TW_WALL_S = int(os.environ.get("RECURSIVE_OPT_TW_WALL_S", "1800"))
    UC14_MIN_TARGET_ITERS = int(os.environ.get("RECURSIVE_OPT_UC14_MIN_ITERS", "5"))
    UC14_OPTIMIZER_CALLS = int(os.environ.get("RECURSIVE_OPT_UC14_OPT_CALLS", "48"))
    UC14_EVAL_CALLS = int(os.environ.get("RECURSIVE_OPT_UC14_EVAL_CALLS", "128"))
    UC14_WALL_S = int(os.environ.get("RECURSIVE_OPT_UC14_WALL_S", "3600"))

    def make_uc14_code_transfer_case():
        train_task = "internal:optimizer_tool_policy"
        target_task = "internal:agentic_trace_policy"
        base = _BASELINES.get("optimizer_tool_policy") or _BASELINES["bbeh_direct_solver"]
        objective = (
            "Learn a reusable optimizer-policy artifact on the source task, then adapt "
            "it to the held-out agentic trace policy. Return concise tools/hints."
        )
        source_prior_fraction = TW_NUM_CANDIDATES / max(1, TW_TOTAL_CANDIDATES)
        spec = {"_component": "optimizer_tool_policy", "_max_examples": MAX_EXAMPLES}
        standard = make_code_arm(
            warm=False, baseline=base, evaluate=evaluate_agentic_trace_policy,
            task_id=target_task, objective=objective, min_target_iters=UC14_MIN_TARGET_ITERS,
        )
        recursive = make_code_arm(
            warm=True, transfer=True, transfer_phase2=True,
            baseline=base, evaluate=evaluate_optimizer_tool_policy, task_id=train_task,
            prior_fraction=source_prior_fraction, objective=objective,
            holdout_task_id=target_task, holdout_evaluate=evaluate_agentic_trace_policy,
            min_target_iters=UC14_MIN_TARGET_ITERS, additive_prior=True,
        )
        return spec, standard, recursive
else:
    print("three-way helpers not loaded (all related switches are False).")


In [ ]:

if enabled("three_way"):
    def run_three_way_code_case(case_name, component_name, baseline, evaluate, task_id, objective, *, max_examples=MAX_EXAMPLES):
        spec = {"_component": component_name, "_max_examples": max_examples}
        standard_runner = make_code_arm(warm=False, baseline=baseline, evaluate=evaluate, task_id=task_id, objective=objective)
        recursive_runner = make_code_arm(warm=True, baseline=baseline, evaluate=evaluate, task_id=task_id, objective=objective)
        report = benchmark_uc(
            case_name,
            initial=spec, standard=spec, recursive=spec,
            output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
            optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
            seeds=TW_SEEDS,
            initial_runner=standard_runner, standard_runner=standard_runner, recursive_runner=recursive_runner,
            notes="code surface cold vs warm two-phase prior",
        ) if LIVE else None
        if report:
            display(Markdown(markdown_report(report)))
        return report

    tw_uc1 = run_three_way_code_case(
        "UC1_code_bbeh_solver", "bbeh_direct_solver", _bbeh_direct_solver,
        make_tracebench_direct_answer_evaluator("internal:multiobjective_bbeh", max_examples=MAX_EXAMPLES, normalizer=_norm_bool_answer),
        "internal:multiobjective_bbeh",
        "Rewrite the Python function to parse BBEH boolean expressions. Input `question` ends with ' is'. Return exactly 'True' or 'False'.",
        max_examples=MAX_EXAMPLES,
    )

    _qasper = HARD_PROMPT_TASKS["qasper"]
    tw_uc2 = benchmark_uc(
        "UC2_prompt_config_qasper",
        initial=config_spec(["starting_artifact"], task=_qasper, family_name="reasoning", inner_steps=2, numeric_constraints=CAUSAL_NUMERIC_CONSTRAINTS),
        standard={**config_spec(["starting_artifact"], task=_qasper, family_name="reasoning", inner_steps=2, numeric_constraints=CAUSAL_NUMERIC_CONSTRAINTS), "reuse_priors": False},
        recursive={**config_spec(["starting_artifact", "batch_design", "batch_size"], task=_qasper, family_name="reasoning", inner_steps=2, numeric_constraints=CAUSAL_NUMERIC_CONSTRAINTS), "reuse_priors": True},
        output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
        optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
        seeds=TW_SEEDS, primary_level="o1_setup",
        notes="recursive = warm prior + active numeric fields vs standard cold prompt-only",
    ) if LIVE else None
    if tw_uc2: display(Markdown(markdown_report(tw_uc2)))

    tw_uc4 = benchmark_uc(
        "UC4_family_policy_prior",
        initial=family_policy_spec("o2", warm=False, targets=CAUSAL_NUMERIC_TARGETS, inner_steps=2, constraints=CAUSAL_NUMERIC_CONSTRAINTS),
        standard={**family_policy_spec("o2", warm=False, targets=CAUSAL_NUMERIC_TARGETS, inner_steps=2, constraints=CAUSAL_NUMERIC_CONSTRAINTS), "reuse_priors": False},
        recursive={**family_policy_spec("o3", warm=True, targets=CAUSAL_NUMERIC_TARGETS, inner_steps=2, constraints=CAUSAL_NUMERIC_CONSTRAINTS), "reuse_priors": True},
        output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
        optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
        seeds=TW_SEEDS, primary_level={"initial": "o2_policy", "standard": "o2_policy", "recursive": "o3_prior"},
        notes="recursive = warm O2->O3 prior transfer vs standard cold single-level policy",
    ) if LIVE else None
    if tw_uc4: display(Markdown(markdown_report(tw_uc4)))

    tw_uc5 = run_three_way_code_case("UC5_tool_policy_code", "optimizer_tool_policy", _baseline_tool_policy, evaluate_optimizer_tool_policy, "internal:optimizer_tool_policy", "Return a compact tools: ... policy selecting only optimizer tools needed by the signal; avoid expensive tools on saturated controls.")
    tw_uc8 = run_three_way_code_case("UC8_campaign_policy_code", "campaign_policy", _baseline_campaign_policy, evaluate_campaign_policy, "internal:campaign_policy", "Rewrite a compact if/elif campaign controller. Return action, task(s), max_examples, and reason.")
    tw_uc9 = run_three_way_code_case("UC9_agentic_policy_code", "agentic_trace_policy", _baseline_agentic_trace_policy, evaluate_agentic_trace_policy, "internal:agentic_trace_policy", "Rewrite a compact policy function. Given a signal string, return tools: ... and hint: ... .")
    tw_uc10 = run_three_way_code_case("UC10_promotion_policy_code", "promotion_policy", _baseline_promotion_policy, evaluate_promotion_policy, "internal:artifact_promotion_policy", "Rewrite a compact artifact gate. Given an artifact_report dict, return action and reason.")
    tw_uc11 = run_three_way_code_case("UC11_prompt_emitter_code", "qasper_prompt_emitter", _qasper_prompt_emitter, make_artifact_emitter_evaluator("hf:qasper", max_examples=HARD_MAX_EXAMPLES, credit_horizon="step"), "hf:qasper", "Rewrite the function so it returns a concise QASPER starting_artifact prompt.", max_examples=HARD_MAX_EXAMPLES)

    _uc13_base = config_spec(["batch_design", "batch_size"], task=FAMILY_TASK, family_name="reasoning", inner_steps=2, numeric_constraints=CAUSAL_NUMERIC_CONSTRAINTS)
    _uc13_recursive = {**_uc13_base, "numeric": {"level_id": "o1_setup", "task": FAMILY_TASK, "fields": ["batch_design", "batch_size"], "optimizer": "optuna", "space": {"batch_design": ("cat", ("random", "failure_balanced", "curriculum", "diversity")), "batch_size": ("cat", (2, 4, 8))}}}
    tw_uc13 = benchmark_uc(
        "UC13_numeric_head_to_head",
        initial=_uc13_base, standard={**_uc13_base, "reuse_priors": False}, recursive=_uc13_recursive,
        output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
        optimizer_llm_calls=TW_OPTIMIZER_CALLS, eval_llm_calls=TW_EVAL_CALLS, wall_time_s=TW_WALL_S,
        seeds=TW_SEEDS, primary_level="o1_setup", recursive_runner=run_numeric_arm,
        notes="recursive/meta numeric optimizer vs generative",
    ) if LIVE else None
    if tw_uc13: display(Markdown(markdown_report(tw_uc13)))

    if enabled("uc14"):
        _uc14_spec, _uc14_std, _uc14_rec = make_uc14_code_transfer_case()
        tw_uc14 = benchmark_uc(
            "UC14_code_transfer",
            initial=_uc14_spec, standard=_uc14_spec, recursive=_uc14_spec,
            output_root=OUTPUT_ROOT, total_candidates=TW_TOTAL_CANDIDATES, num_candidates=TW_NUM_CANDIDATES,
            optimizer_llm_calls=UC14_OPTIMIZER_CALLS, eval_llm_calls=UC14_EVAL_CALLS, wall_time_s=UC14_WALL_S,
            seeds=TW_SEEDS, initial_runner=_uc14_std, standard_runner=_uc14_std, recursive_runner=_uc14_rec,
            notes="recursive source->target held-out code transfer",
        ) if LIVE else None
        if tw_uc14: display(Markdown(markdown_report(tw_uc14)))
else:
    print("three_way skipped.")


In [ ]:

if enabled("guarded_variants"):
    CAMPAIGN_GUARDED_EVALUATOR = GuardedDecisionEvaluator(
        tuple(
            GuardedDecisionCase(
                name=case["name"],
                payload=case["diagnostics"],
                allowed_actions=tuple(sorted(case["actions"])),
                required_targets=tuple(sorted(case["tasks"])) if case["tasks"] else (),
                forbidden_targets=tuple(sorted(case["avoid"])),
                required_terms=tuple(sorted(case["reasons"])),
                numeric_ranges={"max_examples": case["max_examples"]},
                weights={"action": 0.35, "target": 0.20, "avoid": 0.20, "numeric": 0.15, "required": 0.10},
                required_denominator=min(2, len(case["reasons"])),
                hard_forbidden=bool(case["avoid"]),
            )
            for case in CAMPAIGN_POLICY_CASES
        )
    )
    def evaluate_campaign_policy_guarded(component, _task_id):
        return CAMPAIGN_GUARDED_EVALUATOR(component, _task_id)

    uc8_guarded = [("guarded campaign policy", run_code_experiment(
        "campaign_policy_guarded", "internal:campaign_policy_guarded",
        "Rewrite the campaign policy. Return action/task/max_examples/reason and obey hard avoid guards.",
        seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc8_guarded_campaign",
        baseline=_baseline_campaign_policy, evaluate=evaluate_campaign_policy_guarded,
    ))]
    show_table("Use Case 8 variant — guarded campaign policy", uc8_guarded)

    QASPER_FORMAT_CASES = (
        GuardedDecisionCase(name="dataset_verbatim_choice", payload=None, required_terms=("exact", "dataset", "nothing else", "Chinese dataset BIBREF0"), forbidden_terms=("paraphrase", "broader dataset"), weights={"required": 0.75, "forbidden": 0.25}, required_denominator=3, hard_forbidden=True),
        GuardedDecisionCase(name="metric_delta_exactness", payload=None, required_terms=("exact", "MRR", "MR", "Recall@10", "output only"), forbidden_terms=("summarize", "generic"), weights={"required": 0.75, "forbidden": 0.25}, required_denominator=3, hard_forbidden=True),
        GuardedDecisionCase(name="no_wrapper_format", payload=None, required_terms=("no markdown", "no preamble", "final answer"), forbidden_terms=("json object", "explanation", "analysis"), weights={"required": 0.70, "forbidden": 0.30}, required_denominator=2, hard_forbidden=True),
    )
    def evaluate_qasper_prompt_format_guard(component, _task_id):
        return GuardedDecisionEvaluator(QASPER_FORMAT_CASES)(component, _task_id)

    uc11_guarded = [("guarded prompt-format preflight", run_code_experiment(
        "qasper_prompt_format_guard", "internal:qasper_prompt_format_guard",
        "Rewrite the QASPER prompt emitter. Emphasize exact/verbatim answer formats and no wrappers.",
        seeds=DIAGNOSTIC_SEEDS, memory_name="mem_uc11_guarded_prompt_format",
        baseline=_qasper_prompt_emitter, evaluate=evaluate_qasper_prompt_format_guard,
    ))]
    show_table("Use Case 11 variant — guarded prompt-format preflight", uc11_guarded)
else:
    print("guarded_variants skipped.")


In [ ]:

if enabled("stage2"):
    if LIVE:
        uc4_sweep = benchmark_uc_budget_sweep(
            "UC4_family_policy_prior_sweep",
            budget_points=[6, 12],
            initial=family_policy_spec("o2", warm=False, targets=CAUSAL_NUMERIC_TARGETS, inner_steps=2, constraints=CAUSAL_NUMERIC_CONSTRAINTS),
            standard={**family_policy_spec("o2", warm=False, targets=CAUSAL_NUMERIC_TARGETS, inner_steps=2, constraints=CAUSAL_NUMERIC_CONSTRAINTS), "reuse_priors": False},
            recursive={**family_policy_spec("o3", warm=True, targets=CAUSAL_NUMERIC_TARGETS, inner_steps=2, constraints=CAUSAL_NUMERIC_CONSTRAINTS), "reuse_priors": True},
            output_root=OUTPUT_ROOT,
            num_candidates=TW_NUM_CANDIDATES,
            optimizer_llm_calls=TW_OPTIMIZER_CALLS,
            eval_llm_calls=TW_EVAL_CALLS,
            wall_time_s=TW_WALL_S,
            seeds=TW_SEEDS,
            primary_level={"initial": "o2_policy", "standard": "o2_policy", "recursive": "o3_prior"},
        )
        display(Markdown(f"UC4 budget sweep: verdict=`{uc4_sweep['verdict']}` actions={uc4_sweep['actions']} agreed={uc4_sweep['agreed']}"))
        if enabled("uc14"):
            _uc14_spec, _uc14_std, _uc14_rec = make_uc14_code_transfer_case()
            UC14_BUDGET_POINTS = sorted({max(2 * TW_NUM_CANDIDATES, 12), TW_TOTAL_CANDIDATES})
            uc14_sweep = benchmark_uc_budget_sweep(
                "UC14_code_transfer_target_warm_target_heavy",
                budget_points=UC14_BUDGET_POINTS,
                initial=_uc14_spec, standard=_uc14_spec, recursive=_uc14_spec,
                output_root=OUTPUT_ROOT,
                num_candidates=TW_NUM_CANDIDATES,
                optimizer_llm_calls=UC14_OPTIMIZER_CALLS,
                eval_llm_calls=UC14_EVAL_CALLS,
                wall_time_s=UC14_WALL_S,
                seeds=TW_SEEDS,
                initial_runner=_uc14_std, standard_runner=_uc14_std, recursive_runner=_uc14_rec,
                notes="recursive one-source-batch prior -> phase2 held-out target warm start; standard cold target",
            )
            display(Markdown(f"UC14 budget sweep: verdict=`{uc14_sweep['verdict']}` actions={uc14_sweep['actions']} agreed={uc14_sweep['agreed']}"))
    else:
        print("Stage 2 requires LIVE mode.")
else:
    print("stage2 skipped.")


## Tier 1–3 follow-ups

This section is intentionally preserved as an explicit **disabled-by-default** bucket.
When enabled, use the same `benchmark_uc`, `promotion_decision`, and `TW_*` knobs already loaded above.


In [ ]:

if enabled("tier_followups"):
    print("Tier follow-ups are enabled. Add or run the desired Tier cells here.")
else:
    print("tier_followups skipped.")


## Summaries / exports / historical scans


In [ ]:

if enabled("summaries"):
    FINAL_USE_CASES = [
        ("UC1 component code", "uc1"),
        ("UC2 setup/config", "uc2"),
        ("UC3 capability", "uc3"),
        ("UC4 family/transfer", "uc4"),
        ("UC5 optimizer/tool", "uc5"),
        ("UC6 trace feedback", "uc6"),
        ("UC7 graph/suboptimizer", "uc7"),
        ("UC8 campaign policy", "uc8"),
        ("UC9 agentic trace policy", "uc9"),
        ("UC10 promotion policy", "uc10"),
        ("UC11 prompt emitter", "uc11"),
        ("UC12 promoted primitives", "uc12"),
        ("UC13 numeric config", "uc13"),
        ("UC14 code transfer", "uc14"),
    ]
    available = [(label, globals()[var]) for label, var in FINAL_USE_CASES if var in globals() and globals()[var]]
    if available:
        flat = ["| use case | experiment | initial | mean score | delta | std | n | wall_s | eval/trials | best step | artifact version | best artifact file | spec file | notes | best? |", "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---|---|---|---|---|"]
        for uc, data in available:
            best = best_of(data)
            best_label = best[0] if best else None
            for label, result in data:
                scores = _finite(result.get("scores", []))
                mean = statistics.mean(scores) if scores else None
                std = statistics.pstdev(scores) if len(scores) > 1 else None
                delta = (mean - result["initial"]) if mean is not None and result.get("initial") is not None else None
                flat.append(
                    f"| {_md_cell(uc)} | {_md_cell(label)} | {_fmt(result.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                    f"{_fmt(std)} | {len(scores)} | {_fmt(result.get('wall_s'))} | {_fmt_turn(_result_eval_calls(result))} | {_fmt_turn(result.get('best_step'))} | {_fmt_turn(_artifact_version(result))} | "
                    f"{_md_code(result.get('artifact_file') or '-')} | {_md_code(result.get('spec_file') or '-')} | {_md_cell(_notes_for_result(result))} | {'yes' if label == best_label else ''} |"
                )
        _display_markdown("### Final current-kernel rerun table\n" + "\n".join(flat))
        ALL = {label: rows for label, rows in available}
        if enabled("exports"):
            index = export_best_artifacts(ALL)
            _display_markdown("### Exported best artifacts\n" + _artifact_exports_table(index))
    else:
        print("No UC results are available in the current kernel.")
else:
    print("summaries skipped.")


In [ ]:

if enabled("historical"):
    past_runs = summarize_past_runs()
    if past_runs:
        _display_markdown("### Historical runs\n" + past_runs_table(past_runs))
    past_experiments = summarize_past_experiments()
    if past_experiments:
        _display_markdown("### Historical experiments\n" + past_experiments_table(past_experiments, limit=200))
else:
    print("historical scans skipped.")


## How to run this notebook

### Safe default (offline structure / spec inspection)
From the repo root:

```bash
export RECURSIVE_OPT_LIVE=0
jupyter lab examples/recursive_opt_use_cases_structured.ipynb
```

### Real live notebook run
From the repo root:

```bash
export OPENAI_API_KEY=...
export RECURSIVE_OPT_LIVE=1
export RECURSIVE_OPT_MODEL=gpt-5.4-nano
jupyter lab examples/recursive_opt_use_cases_structured.ipynb
```

### Recommended execution order
1. leave `three_way=False`, `stage2=False`, `tier_followups=False`,
2. run configuration + shared helpers,
3. run diagnostics,
4. run UC1–UC13 single-arm cells,
5. inspect summaries / historical scans,
6. only then enable three-way and Stage 2 cells.

### Suggested defaults for the team
- first pass: offline or small live,
- second pass: live UC4 / UC11 / UC13,
- third pass: three-way and UC14 only after the single-arm cells are understood.
